# Thesis figures, one per cell

Each figure below is drawn by its own cell, reading its own data straight out of the committed
result files under `results/accord/`. No cell depends on any variable another figure's cell
created; each one loads what it needs, prints the values it is about to plot, draws, displays the
figure inline, and saves it as a PDF.

Run cells 1 to 3 first (setup, palette, small shared helpers). After that, every figure cell below
runs on its own, in any order.

**No credentialed data is shipped here.** MIMIC-III is a PhysioNet-credentialed dataset; every
result file this notebook reads is an aggregate (counts, rates, confusion-matrix cells, bootstrap
intervals) with no clinical text and no patient or admission identifiers.

## 1. Setup

In [1]:
%matplotlib inline
import os, sys, shutil, json, pathlib, math, statistics
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import norm
from scipy.optimize import brentq
import pandas as pd

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
# REMOVED for custody: this line mounted Google Drive. MIMIC derivatives
# must not sit on cloud-synced storage. See accord/cloud_paths.py.
# REMOVED for custody: this line mounted Google Drive. MIMIC derivatives
# must not sit on cloud-synced storage. See accord/cloud_paths.py.
    DRIVE_BUNDLE = os.environ.get('ACCORD_FIG_OUT', os.path.expanduser('~/mimic_local_only/accord_figures'))
    REPO = "/content/repo"
    os.makedirs(REPO, exist_ok=True)
    dst_results = os.path.join(REPO, "results")
    if os.path.exists(dst_results):
        shutil.rmtree(dst_results)
    shutil.copytree(os.path.join(DRIVE_BUNDLE, "results"), dst_results)
    print(f"copied bundle results: {DRIVE_BUNDLE}/results -> {dst_results}")
else:
    # Local run: this machine already has the repository checked out, so there is
    # nothing to mount or copy. REPO points straight at it.
    REPO = "/Users/rickygole/provenance-encoder"
    print(f"not running in Colab; reading directly from the local repository at {REPO}")

RESULTS = os.path.join(REPO, "results")
RESULTS_ACCORD = os.path.join(RESULTS, "accord")
FIGDIR = os.path.join(REPO, "accord", "thesis", "figures")
# Figure cells that were written as standalone scripts use ROOT; it is the same
# place as REPO, as a Path.
ROOT = pathlib.Path(REPO)
os.makedirs(FIGDIR, exist_ok=True)

# Defensive only: no cell in this notebook resamples or fits anything. Every
# bootstrap interval plotted below was already computed upstream (the seed is
# recorded inside the source JSON, typically 42) and is simply read and plotted.
np.random.seed(42)

n_json = sum(1 for _ in pathlib.Path(RESULTS_ACCORD).rglob("*.json"))
print("REPO           =", REPO)
print("RESULTS_ACCORD =", RESULTS_ACCORD)
print("FIGDIR         =", FIGDIR)
print("json files under results/accord:", n_json)

not running in Colab; reading directly from the local repository at /Users/rickygole/provenance-encoder
REPO           = /Users/rickygole/provenance-encoder
RESULTS_ACCORD = /Users/rickygole/provenance-encoder/results/accord
FIGDIR         = /Users/rickygole/provenance-encoder/accord/thesis/figures
json files under results/accord: 99


## 2. Shared palette

Four colours plus a red accent, used by name in every figure cell below (`DARK`, `MID`, `FILL`,
`DEAD`, `RED`). They differ in **lightness**, not just hue, so the figures still separate correctly
in a black-and-white print; the readout below prints that separation. This is the same palette
already established for this bundle in `Thesis_Figures_Restyle.ipynb`.

In [ ]:
# >>> generated from style.py
# style.py sections 1-6: the ramps, the structure tones, the entity map,
# the type scale, the line weights and the rcParams. Copied verbatim, so
# tools/check_parity.py can assert set-equality with the module itself.
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patheffects as mpe
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch, Rectangle, ArrowStyle

# ---------------------------------------------------------------------------
# 1. THE RAMPS
# ---------------------------------------------------------------------------
class Ramp:
    """One hue, four stops plus BARE. Immutable by convention."""

    __slots__ = ("name", "DEEP", "CORE", "TINT", "WASH", "BARE")

    def __init__(self, name, deep, core, tint, wash, bare):
        self.name = name
        self.DEEP, self.CORE, self.TINT, self.WASH, self.BARE = (
            deep, core, tint, wash, bare)

    def __repr__(self):
        return f"<Ramp {self.name}>"

    def asdict(self):
        return {"DEEP": self.DEEP, "CORE": self.CORE, "TINT": self.TINT,
                "WASH": self.WASH, "BARE": self.BARE}


#                name       DEEP       CORE       TINT       WASH       BARE
COBALT = Ramp("COBALT",  "#0A4DB1", "#0B53C0", "#BDD0EE", "#DEE8F7", "#98B6E4")
AZURE = Ramp("AZURE",    "#20577D", "#3FA9F5", "#A1D5FA", "#D1EBFD", "#78C2F8")
EMERALD = Ramp("EMERALD", "#005F44", "#00A97A", "#98DCC9", "#CEEFE5", "#75D0B7")
AMBER = Ramp("AMBER",    "#714B00", "#F5A300", "#F9C765", "#FCE4B6", "#F6AC1A")
PLUM = Ramp("PLUM",      "#971D6B", "#C3268A", "#EEC2DE", "#F7E1EF", "#E296C6")
SCARLET = Ramp("SCARLET", "#A41704", "#D81E05", "#F5C2BC", "#FAE2DF", "#EE9A8F")

# MAGENTA, TANGERINE and CYAN were the three public models' hues on the
# prevalence grid. The role map below replaced them with neutral tones plus
# marker and line style; the ramps stay only for retired generators.
MAGENTA = Ramp("MAGENTA", "#8E1257", "#E0218A", "#F6C4DF", "#FBE6F1", "#EE8CC0")
TANGERINE = Ramp("TANGERINE", "#7A4200", "#FF8A00", "#FFCC8F", "#FFE9D2", "#FFAE52")
CYAN = Ramp("CYAN",      "#0A5570", "#00A6D6", "#A8DEEF", "#D6F0F8", "#63C6E4")

RAMPS = {r.name: r for r in (COBALT, AZURE, EMERALD, AMBER, PLUM, SCARLET,
                             MAGENTA, TANGERINE, CYAN)}

# Flat names, because a drawing script reads better as SCARLET_DEEP than as
# SCARLET.DEEP inside an f-string-heavy layout block.
COBALT_DEEP, COBALT_CORE, COBALT_TINT, COBALT_WASH, COBALT_BARE = (
    COBALT.DEEP, COBALT.CORE, COBALT.TINT, COBALT.WASH, COBALT.BARE)
AZURE_DEEP, AZURE_CORE, AZURE_TINT, AZURE_WASH, AZURE_BARE = (
    AZURE.DEEP, AZURE.CORE, AZURE.TINT, AZURE.WASH, AZURE.BARE)
EMERALD_DEEP, EMERALD_CORE, EMERALD_TINT, EMERALD_WASH, EMERALD_BARE = (
    EMERALD.DEEP, EMERALD.CORE, EMERALD.TINT, EMERALD.WASH, EMERALD.BARE)
AMBER_DEEP, AMBER_CORE, AMBER_TINT, AMBER_WASH, AMBER_BARE = (
    AMBER.DEEP, AMBER.CORE, AMBER.TINT, AMBER.WASH, AMBER.BARE)
PLUM_DEEP, PLUM_CORE, PLUM_TINT, PLUM_WASH, PLUM_BARE = (
    PLUM.DEEP, PLUM.CORE, PLUM.TINT, PLUM.WASH, PLUM.BARE)
SCARLET_DEEP, SCARLET_CORE, SCARLET_TINT, SCARLET_WASH, SCARLET_BARE = (
    SCARLET.DEEP, SCARLET.CORE, SCARLET.TINT, SCARLET.WASH, SCARLET.BARE)

# ---------------------------------------------------------------------------
# 2. STRUCTURE TONES. No identity, ever.
# ---------------------------------------------------------------------------
PAPER = "#FFFFFF"      # 1.0000
INK = "#16181D"        # 0.0091  17.76:1   text, spines, ticks
GRAPHITE = "#2E3440"   # 0.0341  12.49:1   connectors; a MEASURED non-entity
MUTE = "#5A6072"       # 0.1175   6.27:1   secondary annotation text
SLATE = "#7C8AA5"      # 0.2518   3.48:1   reference lines, seed dots, stubs
MIST = "#DCE3EE"       # 0.7633            neutral structural ground
HAIR = "#EAEDF3"       # 0.8453            the one optional value grid
PLATE_EDGE = "#C4CCDA" # hairline separators; no longer a plate stroke
FOG = "#F2F2F2"        # 0.8879            light neutral region fill

STRUCTURE = {"PAPER": PAPER, "INK": INK, "GRAPHITE": GRAPHITE, "MUTE": MUTE,
             "SLATE": SLATE, "MIST": MIST, "HAIR": HAIR,
             "PLATE_EDGE": PLATE_EDGE, "FOG": FOG}

# The single machine-readable source of every hex in the document.
PALETTE = {name: r.asdict() for name, r in RAMPS.items()}
PALETTE["STRUCTURE"] = dict(STRUCTURE)

# ---------------------------------------------------------------------------
# 3. THE ENTITY MAP. An entity that appears on more than one figure is the same
# hue on every one of them.
# ---------------------------------------------------------------------------
ENTITY_HUE = {
    "gpt4o_zeroshot":       COBALT,
    "gpt4o_fewshot":        AZURE,    # same model, other prompt -> same family
    "accord":               AMBER,    # the detector ensemble
    # The three public models are neutral tones told apart by marker and line
    # style (PUBLIC_MODEL_MARK below), so no public model can be read as the
    # deployed labeller (PLUM) or the ensemble (AMBER). fig_batch2.py's
    # MODEL_HUE must name the same tone; check_figure_invariants.py enforces it.
    "llama3_8b":            INK,
    "weak_labeller":        PLUM,     # drawn in PLUM_DEEP
    "pharmacist_truth":     EMERALD,
    "pretrained_reference": EMERALD,
    "qwen3_8b":             SLATE,
    "granite33_8b":         MUTE,
    "control_model":        EMERALD,
}
# Alpha parameters, the MDE curve, coded-corpus counts: measured, but not an
# entity, so they take the structural measurement tone rather than a hue.
NON_ENTITY_MEASUREMENT = GRAPHITE

# The role map, by name, so a drawing script never chooses a hue for a role.
FAIL = SCARLET                     # failed check or unsupported reading only
WEAK_LABELLER = PLUM.DEEP          # the deployed weak labeller
ENSEMBLE_DEEP, ENSEMBLE_CORE = AMBER.DEEP, AMBER.CORE   # the detector ensemble
# marker, line style, colour for each public model
PUBLIC_MODEL_MARK = {
    "llama3:8b":     ("o", "-", INK),
    "granite3.3:8b": ("s", (0, (4, 2)), MUTE),
    "qwen3:8b":      ("^", (0, (1, 1.5)), SLATE),
}

# The prevalence grid no longer carries hue at all. Its three public models
# are INK, MUTE and SLATE, separated by marker and dash pattern, which survives
# greyscale by construction and keeps hue free for the roles above.

# ---------------------------------------------------------------------------
# 4. TYPE SCALE. Every figure is drawn at its printed size (scale 1.0), so a
# drawn point is a page point.
# ---------------------------------------------------------------------------
T_HEADLINE = 11.0      # INK, regular. A finding, never a title.
T_CLAIM = 22.0         # claim DEEP, bold. Only where a directive says so.
T_DECK = 9.5           # MUTE, regular
T_AXIS = 9.5           # INK
T_TICK = 9.0           # INK
T_SERIES = 9.0         # that series' DEEP
T_NOTE = 8.5           # MUTE, italic
T_PANEL = 10.0         # INK, italic
T_CHIP = 9.0           # PAPER, bold
T_FLOOR = 8.5

# ---------------------------------------------------------------------------
# 5. LINE WEIGHTS. The whole set, no exceptions.
# ---------------------------------------------------------------------------
LW_PLATE = 1.0
LW_SPINE = 0.9
LW_GRID = 0.6
LW_DATA = 1.8
LW_REF = 1.0
LW_INTERVAL = 1.5
LW_NODE = 1.5          # F1 schematic node stroke
LW_CHIP = 1.4          # F2 solid chip rim
LW_CONNECTOR = 1.4
LW_BAR = 1.0           # bar outline, NEW in v3

MS_DOT = 6.6
MS_MARK = 6.2          # marker, non-dot
MS_BARE = 6.6          # open, 1.2pt edge in X_BARE
MS_SEED = 3.2          # SLATE, solid

# The headline's top edge, in points below the top edge of the canvas. It was
# 10.0: a 3.0pt signature bar plus a 7.0pt gap. The bar is gone and the 3.0pt it
# held is given back, so the headline sits 7.0pt from the top and every figure
# starts a little higher on the page than it did.
PAD_HEADLINE = 7.0

REF_DASH = (0, (4, 2.4))
ERRBAR = dict(elinewidth=LW_INTERVAL, capthick=LW_INTERVAL, capsize=3.2)
DOT = dict(markersize=MS_DOT)

# ---------------------------------------------------------------------------
# 6. rcParams
# ---------------------------------------------------------------------------
RCPARAMS = {
    "font.family": "serif",
    # STIXGeneral is Times-metric-compatible and ships inside matplotlib's own
    # mpl-data/fonts/ttf, so it renders identically here and in Colab, which
    # was the entire reason DejaVu was pinned. It is also narrower than DejaVu,
    # so every spacing decision already checked has more room, not less.
    "font.serif": ["STIXGeneral"],
    "mathtext.fontset": "stix",
    "font.size": T_TICK,
    "axes.labelsize": T_AXIS,
    "axes.titlesize": T_AXIS,
    "xtick.labelsize": T_TICK,
    "ytick.labelsize": T_TICK,
    "legend.fontsize": T_TICK,
    "pdf.fonttype": 42,
    # A negative tick label must be a hyphen-minus, not U+2212.
    "axes.unicode_minus": False,
    "text.color": INK,
    "axes.labelcolor": INK,
    "axes.edgecolor": INK,
    "xtick.color": INK,
    "ytick.color": INK,
    # Boxed. All four spines.
    "axes.spines.top": True,
    "axes.spines.right": True,
    "axes.spines.left": True,
    "axes.spines.bottom": True,
    "axes.linewidth": LW_SPINE,
    "xtick.major.width": LW_SPINE,
    "ytick.major.width": LW_SPINE,
    "xtick.major.size": 3.2,
    "ytick.major.size": 3.2,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "axes.grid": False,
    "grid.linewidth": LW_GRID,
    "grid.color": HAIR,
    "lines.linewidth": LW_DATA,
    "legend.frameon": False,
    "figure.dpi": 150,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.02,
}


def setup():
    plt.rcParams.update(RCPARAMS)

setup()
# <<< generated from style.py


## 3. Shared helpers

Small and genuinely reused across every figure cell below: the print-legible style (serif, small
fonts, embedded PDF fonts, no top/right spines), a dashed "chance" line style used by more than one
figure, and a `save_fig` function that writes the PDF and reports its path. Anything used by only
one figure is defined inside that figure's own cell instead, not here.

In [ ]:
# >>> generated from style.py
# style.py sections 7-10: measure, the frame system, marks and grounds,
# and the one interval this set is allowed to compute.
# ---------------------------------------------------------------------------
# 7. MEASURE
# ---------------------------------------------------------------------------
# \textwidth is 433.62pt = 6.014in. Two width classes only.
#
# They were 0.90 and 0.80 of \linewidth. The author asked for every figure to be
# bigger, on ten of the twelve by name, so they are now 0.98 and 0.88: about a
# 9 percent linear increase, and 0.98 rather than 1.00 because a graphic set
# flush to both margins reads as a bleed rather than as a figure. The drawn
# width moves with the printed width and NOT with a scale factor, which is the
# invariant that keeps 9pt type actually 9pt on the page.
TEXTWIDTH_IN = 6.014


def figw(frac):
    """(drawn_width, printed_width) for a given \\linewidth fraction.

    savefig pad is 0.02in per side and canvas() pins the bbox, so
    saved width = drawn + 0.04 = printed exactly.
    """
    printed = frac * TEXTWIDTH_IN
    return printed - 0.04, printed


W98, PRINT_W = figw(0.98)   # 5.8537 drawn, 5.8937 printed
W88, PRINT_W88 = figw(0.88)  # 5.2523 drawn, 5.2923 printed
FIGW = W98                   # the wide class, drawn. Every plot uses it.


def canvas(fig):
    """Pin the saved bounding box to the whole figure.

    savefig crops to the artists, so the width of the file would otherwise
    depend on how far the longest annotation happened to reach and no two
    figures would come out the same measure.
    """
    fig.patches.append(Rectangle((0, 0), 1, 1, transform=fig.transFigure,
                                 fill=False, linewidth=0, edgecolor="none"))


# ---------------------------------------------------------------------------
# 8. THE FRAME SYSTEM. Two enclosures, exactly one per figure.
# ---------------------------------------------------------------------------
def frame(ax, grid=False):
    """Full boxed axes. Replaces v2's lean().

    All four spines at 0.9pt INK, ticks outward. set_bounds() clipping is
    removed everywhere: a closed box cannot be clipped and the two ideas fight.

    grid draws the one optional value grid: True for both axes, or "x" / "y"
    for the one axis the plot is read along. The rule the builder does NOT get
    to decide: a plot read for VALUES gets the grid; a plot read for position
    or shape does not. It is specified per figure.
    """
    for s in ("top", "right", "left", "bottom"):
        ax.spines[s].set_visible(True)
        ax.spines[s].set_linewidth(LW_SPINE)
        ax.spines[s].set_color(INK)
    if grid:
        ax.set_axisbelow(True)
        ax.grid(True, axis="both" if grid is True else grid, color=HAIR,
                linewidth=LW_GRID, zorder=0)
    return ax


def plate(fig):
    """The schematic enclosure: a rounded rectangle covering the full canvas,
    inset 1.5pt from every edge, corner radius 4pt, PAPER fill, PLATE_EDGE
    stroke at 1.0pt. No inner axes box. A plot never gets one."""
    w_in, h_in = fig.get_size_inches()
    ix, iy = 1.5 / (72 * w_in), 1.5 / (72 * h_in)
    p = FancyBboxPatch(
        (ix, iy), 1 - 2 * ix, 1 - 2 * iy,
        boxstyle="round,pad=0,rounding_size=0.010",
        transform=fig.transFigure, facecolor=PAPER, edgecolor=PLATE_EDGE,
        linewidth=LW_PLATE, mutation_aspect=w_in / h_in, zorder=-10,
        clip_on=False)
    fig.patches.append(p)
    return p


def signature(fig, hue_deep, headline=None, deck=None, headline_x=None):
    """The headline block: one line stating what the figure FOUND, never what
    it plots, at T_HEADLINE, so the figure stops reading as an
    inclusion, plus an optional 9.5pt MUTE deck under it.

    It used to draw a 3.0pt bar in hue_deep flush to the top edge as well. That
    bar is gone by the author's instruction, on all twelve figures rather than
    on the seven they named, because half a set with a coloured rule and half
    without is worse than either. hue_deep stays in the signature and is
    deliberately unused: every call site names the figure's dominant hue, that
    naming is the one place each figure declares which hue it is built on, and
    deleting the argument would delete twelve statements of intent to save
    twelve tokens. The 3.0pt the bar held is given back to the headline, which
    now sits PAD_HEADLINE from the top instead of 10.0.
    """
    h_in = fig.get_size_inches()[1]
    y = 1 - PAD_HEADLINE / (72 * h_in)
    x = 0.012 if headline_x is None else headline_x
    if headline:
        fig.text(x, y, headline, ha="left", va="top", fontsize=T_HEADLINE,
                 color=INK, zorder=21)
        # Advance by the headline's real height. A headline may be two lines,
        # and advancing one line height put the deck on top of the second.
        n_lines = headline.count("\n") + 1
        y -= (T_HEADLINE * 1.06 * n_lines) / (72 * h_in)
    if deck:
        fig.text(x, y, deck, ha="left", va="top", fontsize=T_DECK, color=MUTE,
                 zorder=21)
    return y


def panel_label(fig, ax, text):
    """Italic 10pt INK in FIGURE coordinates: x at the axes box's left edge,
    y 5pt above the axes box top edge. Always outside the frame, never inside,
    never in data coordinates. No figure computes this itself."""
    h_in = fig.get_size_inches()[1]
    p = ax.get_position()
    return fig.text(p.x0, p.y1 + 5.0 / (72 * h_in), text, ha="left",
                    va="bottom", fontsize=T_PANEL, style="italic", color=INK,
                    zorder=21)


# ---------------------------------------------------------------------------
# 9. MARKS AND GROUNDS
# ---------------------------------------------------------------------------
# matplotlib's head_width is a HALF-width, and the 1.4pt stroke adds about
# 1.9pt to the rendered width, so the spec's literal (0.62, 0.30) at
# mutation_scale 12 renders 7.2pt long by 9.4pt wide and misses the 6.0-8.5pt
# width gate. These two numbers are the ones that hit the spec's stated target
# of about 7.4pt long by 7.2pt wide: measured 7.2 x 7.6 at scale 12.
_ARROWSTYLE = ArrowStyle("-|>", head_length=0.62, head_width=0.24)


def arrow(ax, xy_from, xy_to, color=GRAPHITE, lw=LW_CONNECTOR, zorder=3,
          mutation_scale=12, **kw):
    """One arrowhead for the whole set. matplotlib's default micro-heads are
    replaced everywhere: this targets a head about 7.4pt long and 7.2pt wide,
    solid-filled in the stroke colour. No script builds one by hand."""
    a = FancyArrowPatch(xy_from, xy_to, arrowstyle=_ARROWSTYLE,
                        mutation_scale=mutation_scale, lw=lw, shrinkA=0,
                        shrinkB=0, joinstyle="miter", capstyle="butt",
                        color=color, zorder=zorder, **kw)
    ax.add_patch(a)
    return a


def chip(ax, x, y, w, h, hue, text, fontsize=T_CHIP, radius=3.0, zorder=6,
         ha="center", stroke=None):
    """F2 SOLID CHIP: X_DEEP fill, PAPER text, bold, no rim.

    hue may be a Ramp or a (fill, rim) pair, so the SLATE/MIST no-verdict state
    can use the same helper without inventing a seventh ramp. The rim is kept in
    the signature and ignored by default: a saturated fill on paper needs no
    outline to hold its shape, and the 1.4pt CORE rim it used to carry made a
    small chip read as a button. Pass stroke= to draw one deliberately.
    """
    if isinstance(hue, Ramp):
        fill, rim = hue.DEEP, hue.CORE
    else:
        fill, rim = hue
    rim = stroke                      # None unless a caller asks for one
    ax.add_patch(FancyBboxPatch(
        (x, y), w, h,
        boxstyle=f"round,pad=0,rounding_size={radius}", mutation_aspect=1.0,
        linewidth=(LW_CHIP if rim else 0), edgecolor=(rim or "none"),
        facecolor=fill, zorder=zorder,
        clip_on=False))
    tx = x + w / 2 if ha == "center" else x + 6.0
    t = ax.text(tx, y + h / 2, text, ha=ha, va="center", fontsize=fontsize,
                color=PAPER, fontweight="bold", zorder=zorder + 1,
                clip_on=False)
    return t


def node(ax, x, y, w, h, hue_tint, hue_deep, radius=3.0, zorder=4, lw=LW_NODE):
    """F1 TINT PANEL: X_TINT fill, X_DEEP stroke 1.5pt. Text is drawn by the
    caller in INK, because a node's text is sometimes two lines at two sizes."""
    p = FancyBboxPatch(
        (x, y), w, h,
        boxstyle=f"round,pad=0,rounding_size={radius}", mutation_aspect=1.0,
        linewidth=lw, edgecolor=hue_deep, facecolor=hue_tint, zorder=zorder,
        clip_on=False)
    ax.add_patch(p)
    return p


def band(ax, x0, x1, tint, zorder=0, axis="x"):
    """F3 WASH/TINT GROUND spanning the full extent of the other axis. No
    stroke, no alpha: the tint hex is precomputed."""
    if axis == "x":
        return ax.axvspan(x0, x1, facecolor=tint, edgecolor="none",
                          linewidth=0, zorder=zorder)
    return ax.axhspan(x0, x1, facecolor=tint, edgecolor="none", linewidth=0,
                      zorder=zorder)


def case(artist, linewidth=2.8):
    """Casing: any text or line sitting on a filled field takes a white stroke
    under it, so it reads against the fill and against the paper both. Applies
    over TINT, WASH and heatmap fields only. Text on a SOLID chip is white and
    is never cased."""
    fx = [mpe.withStroke(linewidth=linewidth, foreground=PAPER)]
    try:
        artist.set_path_effects(fx)
    except AttributeError:
        for a in artist:
            a.set_path_effects(fx)
    return artist


def cased(fontsize=T_SERIES, linewidth=2.8):
    """Kwargs form, for the common `ax.text(..., **style.cased())` call."""
    return dict(fontsize=fontsize,
                path_effects=[mpe.withStroke(linewidth=linewidth,
                                             foreground=PAPER)])


# ---------------------------------------------------------------------------
# 10. STATISTICS
# ---------------------------------------------------------------------------
def wilson_ci(k, n, z=1.959963984540054):
    """Wilson score interval for a binomial proportion. Closed-form and
    deterministic: it needs the two counts and nothing else, which is why it is
    the only interval this set is allowed to add to a figure that does not
    already carry one."""
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    denom = 1 + z**2 / n
    centre = p + z**2 / (2 * n)
    adj = z * ((p * (1 - p) / n + z**2 / (4 * n**2)) ** 0.5)
    # Clamped to [0, 1] and to the estimate itself. At k = 0 the exact lower
    # bound is 0 and the arithmetic lands a few parts in 10^18 either side of
    # it, which is not a real interval but is enough to make a plotting library
    # refuse a negative error bar.
    lo = min(p, max(0.0, (centre - adj) / denom))
    hi = max(p, min(1.0, (centre + adj) / denom))
    return (lo, hi)


# Notebook-local bindings the drawing cells below expect. They exist in the
# scripts as module globals and are reproduced here so a figure cell is the
# same code in both places.
import statistics as st, glob
ROOT = pathlib.Path(REPO)
HERE = pathlib.Path(FIGDIR)
SUPP = pathlib.Path(RESULTS_ACCORD) / "decontam"
MANIFEST = []
man = {}


def load(*parts):
    path = os.path.join(RESULTS_ACCORD, *parts)
    with open(path) as f:
        return json.load(f), path


def register(fname, label, sources, note=""):
    MANIFEST.append({"file": os.path.join("figures", fname), "label": label,
                     "sources": sources, "note": note})


def mcc_from_counts(tp, tn, fp, fn):
    num = tp * tn - fp * fn
    den = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    return 0.0 if den == 0 else num / den
# <<< generated from style.py


## 4. The data these figures are built from

Every figure below reads its numbers out of a stored result file. Nothing is
typed in from a draft, a note or from memory. This cell lists each file, checks
it is present, and shows its top-level keys, so the source material is visible
before a single figure is drawn.

If a file is missing here, the figure that depends on it will fail loudly rather
than fall back to a literal.

In [4]:
SOURCES = [
    ("results/accord/exemplar/exemplar_correctness_recomputed.json", "Fig 1, 4"),
    ("results/accord/diag/step0_1_join_and_calibration.json",        "Fig 1, 2, 5"),
    ("results/accord/diag/step2_signal_check.json",                  "Fig 1, 3, 5"),
    ("results/accord/decontam/corrected_gold_metrics.json",          "Fig 1, 8, 10"),
    ("results/accord/reconciliation_c28.json",                       "Fig 2"),
    ("results/accord/stats/cohort_icd_counts.json",                  "Fig 3"),
    ("results/accord/generality/transfer_results_correct.json",      "Fig 3, 6"),
    ("results/accord/VERIFIED_FACTS.json",                           "Fig 3"),
    ("results/accord/prevalence_replication/sweep_results.json",     "Fig 5"),
    ("results/accord/decontam/accord_v2_decontam_gold_aggregate.json","Fig 6"),
    ("results/accord/section6_auroc_power.json",                     "Fig 7"),
    ("results/task_runs.json",                                       "Fig 7"),
    ("results/accord/stats/intervals.json",                          "Fig 8"),
    ("results/accord/survey/coding_summary.json",                    "Fig 9"),
    ("results/accord/control/final_report.json",                     "Fig 8, 10"),
    ("results/accord/scaling/scaling_results.json",                  "Fig 11"),
    ("results/accord/selection/threshold_robustness_results.json",   "Fig 12"),
    ("results/accord/diag/step3_baselines.json",                     "Fig 14"),
    ("results/accord/differential/exp1a_gold_error_rates_by_group.json", "Fig 2, 12"),
    ("results/accord/number_provenance_audit.json",                  "provenance record"),
]

print(f"{'file':66s} {'kB':>6}  used by")
print("-" * 100)
missing = []
for rel, used in SOURCES:
    path = os.path.join(REPO, rel)
    if not os.path.exists(path):
        print(f"{rel:66s} {'--':>6}  MISSING")
        missing.append(rel)
        continue
    kb = os.path.getsize(path) / 1024
    print(f"{rel:66s} {kb:6.1f}  {used}")

print()
if missing:
    print(f"{len(missing)} of {len(SOURCES)} files are missing. The figures that read them will")
    print("fail rather than substitute a value. Check that the Drive bundle copied correctly.")
else:
    print(f"all {len(SOURCES)} source files present")

print()
print("top-level keys of each file")
print("-" * 100)
for rel, _ in SOURCES:
    path = os.path.join(REPO, rel)
    if not os.path.exists(path):
        continue
    try:
        d = json.load(open(path))
    except Exception as e:
        print(f"{rel}: could not parse ({e})")
        continue
    keys = list(d.keys()) if isinstance(d, dict) else [f"(list of {len(d)})"]
    shown = ", ".join(str(k) for k in keys[:6])
    more = f"  ... and {len(keys) - 6} more" if len(keys) > 6 else ""
    print(f"{rel.split('/')[-1]:52s} {shown}{more}")


file                                                                   kB  used by
----------------------------------------------------------------------------------------------------
results/accord/exemplar/exemplar_correctness_recomputed.json          1.4  Fig 1, 4
results/accord/diag/step0_1_join_and_calibration.json                 4.2  Fig 1, 2, 5
results/accord/diag/step2_signal_check.json                           2.0  Fig 1, 3, 5
results/accord/decontam/corrected_gold_metrics.json                   1.1  Fig 1, 8, 10
results/accord/reconciliation_c28.json                               12.8  Fig 2
results/accord/stats/cohort_icd_counts.json                           0.4  Fig 3
results/accord/generality/transfer_results_correct.json               0.9  Fig 3, 6
results/accord/VERIFIED_FACTS.json                                  220.3  Fig 3
results/accord/prevalence_replication/sweep_results.json             16.8  Fig 5
results/accord/decontam/accord_v2_decontam_gold_aggregate.json

## Figure 1: `fig-pipeline-status`


In [5]:
p_exemplar = os.path.join(RESULTS_ACCORD, "exemplar", "exemplar_correctness_recomputed.json")
d_exemplar = json.load(open(p_exemplar))
ex = d_exemplar["rule_1_2_pooled"]
defect_rate = ex["n_wrong"] / ex["n_resolved"]

p_step01 = os.path.join(RESULTS_ACCORD, "diag", "step0_1_join_and_calibration.json")
d_step01 = json.load(open(p_step01))
xtab = d_step01["weak_confidence_is_degenerate"]["crosstab_label_x_confidence"]
weak_pos = xtab["0.95"]["contradiction"]
weak_neg = xtab["1.0"]["not_contradiction"]
weak_pos_rate = weak_pos / (weak_pos + weak_neg)

p_step2 = os.path.join(RESULTS_ACCORD, "diag", "step2_signal_check.json")
d_step2 = json.load(open(p_step2))
pharm_rate_all = d_step2["contra_prob_vs_pharm_label__ambiguous_as_negative_n299"]["base_rate"]

# One multiplier for the whole thesis, and it is the gold-set one: the weak
# labeller's marginal on the 299 annotated pairs over the pharmacist's base
# rate on those same 299 pairs. Both terms are measured on the same rows, which
# is the only version of this ratio that compares like with like. This figure
# used to divide the training-pool marginal by the gold base rate, a different
# denominator on each side of the division, which gave a second slightly
# different headline number that the introduction then spent a sentence in two
# captions explaining away.
p_exp1a = os.path.join(RESULTS_ACCORD, "differential", "exp1a_gold_error_rates_by_group.json")
d_exp1a = json.load(open(p_exp1a))
gold_weak_pos = sum(g["TP"] + g["FP"] for g in d_exp1a["language_group3"])
gold_n = sum(g["n"] for g in d_exp1a["language_group3"])
ratio = (gold_weak_pos / gold_n) / pharm_rate_all

p_decontam = os.path.join(RESULTS_ACCORD, "decontam", "corrected_gold_metrics.json")
d_decontam = json.load(open(p_decontam))
auroc = d_decontam["aggregate"]["ensemble_auroc"]
ci = d_decontam["aggregate"]["ensemble_ci95"]

print("fig-pipeline-status: values plotted")
print(f"  exemplar defect rate, rule_1_2_pooled     : {ex['n_wrong']}/{ex['n_resolved']} = {defect_rate:.4f}")
print(f"  weak-label positive rate, train pool       : {weak_pos}/{weak_pos + weak_neg} = {weak_pos_rate:.4f}")
print(f"  weak-label positive rate, gold set          : {gold_weak_pos}/{gold_n} = {gold_weak_pos / gold_n:.4f}")
print(f"  pharmacist base rate, n=299 (ambig=neg)     : {pharm_rate_all:.4f}")
print(f"  prevalence mismatch ratio, gold set on gold set: {ratio:.1f}x")
print(f"  decontaminated 3-seed ensemble AUROC         : {auroc:.4f}, 95% CI [{ci[0]:.3f}, {ci[1]:.3f}]")
print("  two categorical, unsourced-by-number badges (stage 'Detector training'): see markdown above")

# Six evenly spaced columns. Every string below is wrapped to at most about
# twelve characters so that no badge and no diagnostic can grow wider than
# its column: the previous version wrapped to the natural phrase length and
# the long badges collided with their neighbours and with the row beneath.

fig-pipeline-status: values plotted
  exemplar defect rate, rule_1_2_pooled     : 65/137 = 0.4745
  weak-label positive rate, train pool       : 1425/1679 = 0.8487
  weak-label positive rate, gold set          : 261/299 = 0.8729
  pharmacist base rate, n=299 (ambig=neg)     : 0.0401
  prevalence mismatch ratio, gold set on gold set: 21.8x
  decontaminated 3-seed ensemble AUROC         : 0.5145, 95% CI [0.321, 0.703]
  two categorical, unsourced-by-number badges (stage 'Detector training'): see markdown above


**Step 2. The figure.** Drawn only from the values printed above, then written to `fig-pipeline-status.pdf`.

In [ ]:
# ROUND 7 REBUILD (fixes_r7 N9). The previous canvas numbered six stages,
# headlined "four of six stages hid a failure", drew evaluation as a
# failure, credited "three ablation claims" that no ablation ever tested and
# checked the architecture with an OOD transfer F1 the thesis does not
# interpret. This one draws the thesis's own four stages, numbered 1 to 4
# as Chapter 1 numbers them. Stages 1 to 3 each reported a number that its
# control or code audit does not support the reading of that number;
# stage 4 is a bounded null, and
# the two states are told apart by fill (solid chip against outlined chip)
# and by a word, so the distinction survives greyscale.
#
# The candidate chip is a RANGE, never a rate: no instrument establishes
# how often the positional exemplar misses (C106). The low end is one
# minus the step2plus join's pooled correct rate; the high end is the
# unvalidated direct extractor.
base = fact("gold.base_rate_299")
n_gold = fact("gold.n_total")
ex_lo = 1.0 - fact("exemplar.rule_1_2_pooled_exemplar_correct_rate_in_299.rate")
ex_hi = fact("exemplardirect.defect_rate")
ex_mid = fact("exemplar_recomputed.rule_1_2_pooled.wrong_rate")
assert ex_lo < ex_mid < ex_hi
n_c120 = fact("c120.g299.n")
n_flag = fact("c120.g299.n_flagged")
mcc = fact("c120.g299.mcc")
mcc_ceil = fact("c120.g299.ceiling_mcc")
assert n_c120 == n_gold
f1_rule = fact("rulehead.test_rule_type_macro_f1")
auroc = fact("decontam.aggregate.ensemble_auroc")
ci_hi = fact("decontam.aggregate.ensemble_ci95_hi")
floor = fact("sizectl108.power_curves.primary_crossing_power_0_80")
# The bounded-null reading only holds while the interval's upper end sits
# under the floor; if either moves past the other the chip would lie.
assert ci_hi < floor, "interval upper bound no longer under the C108 floor"

# ROUND 8 (committee). Stage 1 is "not established", not "unsupported":
# three instruments disagree on the exemplar defect (0, 0.4745, 0.9315), so
# no check rejects the reported precision; none establishes it either. The
# three states are told apart in greyscale by chip value alone: solid
# scarlet (dark) for unsupported, a SCARLET_TINT fill under a dashed rim
# (mid grey) for not established, paper under a solid rim for bounded null.
NOTEST, UNSUP, BOUNDED = "not established", "unsupported", "bounded null"
# ROUND 9 (committee). The numbers largely stand; what failed is the
# reading of each. The per-stage label for an unsupported stage says so.
LABEL = {NOTEST: NOTEST, UNSUP: "reading unsupported", BOUNDED: BOUNDED}
rows = [
    (1, "Candidate mining", NOTEST,
     f"precision {base:.4f}\non {n_gold} gold pairs",
     "Is the exemplar in the trigger set?",
     f"miss rate {ex_lo:.0f}, {ex_mid:.4f} or {ex_hi:.4f} by instrument"),
    (2, "Weak supervision", UNSUP,
     f"{n_flag} of {n_c120} pairs\nflagged",
     "Marginal ceiling, then measured agreement",
     f"MCC {mcc:.2f}, ceiling {mcc_ceil:.2f}"),
    (3, "Architecture", UNSUP,
     f"rule-type F1 {f1_rule:.4f},\ntwo more components",
     "Code audit of the three credited components",
     "artifact, target in input, untrained"),
    (4, "Evaluation", BOUNDED,
     f"AUROC {auroc:.2f}",
     "Interval upper end against the power floor",
     f"upper {ci_hi:.2f}, floor {floor:.2f}"),
]

W, H = FIGW, 3.10
fig, ax = plt.subplots(figsize=(W, H))
canvas(fig)
fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
ax.set_xlim(0, W * 72)
ax.set_ylim(0, H * 72)
ax.axis("off")
fig.canvas.draw()
renderer = fig.canvas.get_renderer()

def width_pt(text, **kw):
    probe = ax.text(0, 0, text, **kw)
    w = probe.get_window_extent(renderer).width * 72.0 / fig.dpi
    probe.remove()
    return w

row_mid = {1: 150.0, 2: 108.0, 3: 66.0, 4: 24.0}
# A narrow spine: the rail, the stage numbers and the stage names take the
# left fifth of the canvas, and the columns are placed from measured text
# widths rather than fixed offsets.
X_SPINE, X_NUM, X_NAME = 15.0, 32.0, 42.0
name_w = max(max(width_pt(r[1], fontsize=10.5) for r in rows),
             max(width_pt(LABEL[r[2]], fontsize=T_NOTE, style="italic")
                 for r in rows))
X_REP = X_NAME + name_w + 14.0
rep_w = max(width_pt(r[3], fontsize=T_SERIES, linespacing=1.22) for r in rows)
X_CHK = X_REP + rep_w + 16.0

# The carrier: stages 2 to 4 consumed the exemplar stage 1 selected. It
# records shared input, not cause and not a defect rate, so it is a neutral
# rule and not scarlet (scarlet marks a failed check only).
ax.add_patch(mpatches.Circle((X_SPINE, row_mid[1] + 5.5), 3.0,
                             facecolor=GRAPHITE, edgecolor="none",
                             zorder=6))
ax.plot([X_SPINE, X_SPINE], [row_mid[1] + 5.5, row_mid[4] + 5.5],
        color=GRAPHITE, lw=1.4, zorder=4, solid_capstyle="butt")
for n in (2, 3, 4):
    arrow(ax, (X_SPINE, row_mid[n] + 5.5), (X_NUM - 7.0, row_mid[n] + 5.5),
          color=GRAPHITE, lw=1.2, mutation_scale=9)
ax.text(X_SPINE - 6.5, (row_mid[2] + row_mid[4]) / 2 + 5.5,
        "the same selected exemplar", rotation=90, ha="center",
        va="center", fontsize=T_NOTE, style="italic", color=MUTE,
        zorder=5)

for n, name, status, reported, check, finding in rows:
    mid = row_mid[n]
    ax.text(X_NUM, mid + 5.5, str(n), ha="center", va="center",
            fontsize=10.0, color=INK, zorder=5)
    ax.text(X_NAME, mid + 5.5, name, ha="left", va="center", fontsize=10.5,
            color=INK, zorder=5)
    ax.text(X_NAME, mid - 8.0, LABEL[status], ha="left", va="center",
            fontsize=T_NOTE, style="italic",
            color=SCARLET_DEEP if status == UNSUP else MUTE, zorder=5)
    ax.text(X_REP, mid, reported, ha="left", va="center",
            fontsize=T_SERIES, color=INK, linespacing=1.22, zorder=5)
    ax.text(X_CHK, mid + 9.5, check, ha="left", va="center",
            fontsize=T_NOTE, style="italic", color=MUTE, zorder=5)
    w_i = width_pt(finding, fontsize=T_CHIP, fontweight="bold") + 12.0
    # Three chip encodings that stay distinct in greyscale: a solid dark
    # scarlet fill for unsupported, a light MIST fill under a dashed rim
    # for not established, bare paper under a solid rim for bounded null.
    if status == UNSUP:
        chip(ax, X_CHK, mid - 14.5, w_i, 13.0, SCARLET, finding,
             fontsize=T_CHIP, radius=3.0, ha="left")
    elif status == NOTEST:
        p = node(ax, X_CHK, mid - 14.5, w_i, 13.0, MIST, GRAPHITE,
                 radius=3.0, zorder=6, lw=1.2)
        p.set_linestyle((0, (3.0, 1.8)))
        ax.text(X_CHK + 6.0, mid - 8.0, finding, ha="left", va="center",
                fontsize=T_CHIP, fontweight="bold", color=INK, zorder=7)
    else:
        node(ax, X_CHK, mid - 14.5, w_i, 13.0, PAPER, GRAPHITE,
             radius=3.0, zorder=6, lw=1.4)
        ax.text(X_CHK + 6.0, mid - 8.0, finding, ha="left", va="center",
                fontsize=T_CHIP, fontweight="bold", color=INK, zorder=7)
    assert X_CHK + w_i < W * 72 - 4.0, f"chip overflows the canvas: {finding}"

ax.text(X_REP, 178.0, "REPORTED", ha="left", va="center",
        fontsize=T_FLOOR, color=MUTE, zorder=5)
ax.text(X_CHK, 178.0, "CHECK AND FINDING", ha="left", va="center",
        fontsize=T_FLOOR, color=MUTE, zorder=5)

signature(fig, SCARLET_DEEP,
          headline="No stage's reported reading survived its check: one not "
          "established,\ntwo unsupported, one a bounded null.")
fname = "fig-pipeline-status.pdf"
fig.savefig(os.path.join(FIGDIR, fname))
plt.close(fig)
srcs = [_FACTS_PATH] + sorted({fact_source(k) for k in PIPELINE_KEYS} - {_FACTS_PATH})
register(fname, "fig:pipeline-status", srcs,
         "Four stages numbered 1 to 4 as Chapter 1 numbers them. Every number "
         "is read by key from results/accord/VERIFIED_FACTS.json (keys: "
         + ", ".join(PIPELINE_KEYS) + "); no withdrawn key is readable. AUROC "
         "and MCC print to two decimals on the canvas; rates keep four. Three "
         "chip encodings, distinct in greyscale: stage 1 not established (MIST "
         "fill, dashed GRAPHITE rim; miss rate 0, 0.4745 or 0.9315 by "
         "instrument); stages 2 and 3 reading unsupported (solid scarlet, the "
         "only scarlet on the canvas: MCC 0.03 against a ceiling of 0.08; the "
         "three credited architectural components fail a code audit); stage 4 "
         "bounded null (paper fill, solid rim: interval upper end 0.70 under the "
         "80 percent power floor 0.75). The carrier rail is GRAPHITE, not "
         "scarlet: it records that stages 2 to 4 consumed the exemplar stage 1 "
         "selected, which is shared input, not a failed check. No plate; the "
         "stage column is sized from measured text widths so the spine takes "
         "about the left fifth of the canvas. No internal identifiers on the "
         "canvas.")


## Figure 2: `fig-intro-headline`


In [7]:
p_step01 = os.path.join(RESULTS_ACCORD, "diag", "step0_1_join_and_calibration.json")
d_step01 = json.load(open(p_step01))
xtab = d_step01["weak_confidence_is_degenerate"]["crosstab_label_x_confidence"]
weak_pos, weak_neg = xtab["0.95"]["contradiction"], xtab["1.0"]["not_contradiction"]
weak_n = weak_pos + weak_neg
weak_rate = weak_pos / weak_n

p_exp1a = os.path.join(RESULTS_ACCORD, "differential", "exp1a_gold_error_rates_by_group.json")
d_exp1a = json.load(open(p_exp1a))
lang = d_exp1a["language_group3"]
pred_pos = sum(g["TP"] + g["FP"] for g in lang)
pred_n = sum(g["n"] for g in lang)
pred_rate = pred_pos / pred_n

p_step2 = os.path.join(RESULTS_ACCORD, "diag", "step2_signal_check.json")
d_step2 = json.load(open(p_step2))
d299 = d_step2["contra_prob_vs_pharm_label__ambiguous_as_negative_n299"]
pharm_pos, pharm_n = d299["n_pos"], d299["n"]
pharm_rate = pharm_pos / pharm_n

p_intervals = os.path.join(RESULTS_ACCORD, "stats", "intervals.json")
d_intervals = json.load(open(p_intervals))
mcc = d_intervals["E2_mcc_intervals"]
order = ["gpt4o_zeroshot", "gpt4o_fewshot", "accord_binary_at_0.5_253core", "llama_baseline"]
labels = ["GPT-4o zero-shot", "GPT-4o few-shot", "ACCORD (contam.)", "Llama 3 8B"]

p_c28 = os.path.join(RESULTS_ACCORD, "reconciliation_c28.json")
d_c28 = json.load(open(p_c28))
accord_pred = d_c28["a_what_each_number_counts"]["278"]
weak_marginal = d_c28["a_what_each_number_counts"]["261"]

def wilson_ci(k, n, z=1.959963984540054):
    """Wilson score interval for a binomial proportion; closed-form, deterministic."""
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    denom = 1 + z ** 2 / n
    centre = p + z ** 2 / (2 * n)
    adj = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2))
    # Clamped to [0, 1] and to the estimate itself, identically to the helper in
    # the setup cell: at k = 0 the exact lower bound is 0 and the arithmetic
    # lands a few parts in 10^18 either side of it.
    lo = min(p, max(0.0, (centre - adj) / denom))
    hi = max(p, min(1.0, (centre + adj) / denom))
    return (lo, hi)

print("fig-intro-headline: values plotted")
print(f"  weak-label positive rate, train pool     : {weak_pos}/{weak_n} = {weak_rate:.4f}")
print(f"  left-panel middle bar (exp1a TP+FP sum)  : {pred_pos}/{pred_n} = {pred_rate:.4f}  (the weak labeller on the gold set, NOT ACCORD)")
print(f"  pharmacist true rate, gold set (n=299)   : {pharm_pos}/{pharm_n} = {pharm_rate:.4f}")
for k in order:
    v = mcc[k]
    print(f"  MCC {k:32s}: {v['mcc']:+.4f}  95% CI [{v['ci95'][0]:+.4f}, {v['ci95'][1]:+.4f}]")
print(f"  reconciliation_c28: ACCORD predicted-positive = {accord_pred['value']}/{accord_pred['denominator']}"
      f"  (accord_pred == 'contradiction')")
print(f"  reconciliation_c28: weak-labeller marginal    = {weak_marginal['value']}/{weak_marginal['denominator']}"
      f"  (weak_label == 'contradiction', matches the {pred_pos}/{pred_n} bar above)")

# The right panel gets the wider share: its four category labels need room
# side by side, while the left panel's y labels live outside its axes.

fig-intro-headline: values plotted
  weak-label positive rate, train pool     : 1425/1679 = 0.8487
  left-panel middle bar (exp1a TP+FP sum)  : 261/299 = 0.8729  (the weak labeller on the gold set, NOT ACCORD)
  pharmacist true rate, gold set (n=299)   : 12/299 = 0.0401
  MCC gpt4o_zeroshot                  : +0.1089  95% CI [-0.0319, +0.2484]
  MCC gpt4o_fewshot                   : +0.0117  95% CI [-0.1120, +0.1319]
  MCC accord_binary_at_0.5_253core    : -0.0070  95% CI [-0.1626, +0.0753]
  MCC llama_baseline                  : -0.0436  95% CI [-0.2337, +0.0563]
  reconciliation_c28: ACCORD predicted-positive = 278/299  (accord_pred == 'contradiction')
  reconciliation_c28: weak-labeller marginal    = 261/299  (weak_label == 'contradiction', matches the 261/299 bar above)


**Step 2. The figure.** Drawn only from the values printed above, then written to `fig-intro-headline.pdf`.

In [ ]:
step01, p_step01 = load("diag", "step0_1_join_and_calibration.json")
xtab = step01["weak_confidence_is_degenerate"]["crosstab_label_x_confidence"]
weak_pos, weak_neg = xtab["0.95"]["contradiction"], xtab["1.0"]["not_contradiction"]
weak_n = weak_pos + weak_neg
weak_rate = weak_pos / weak_n

exp1a, p_exp1a = load("differential", "exp1a_gold_error_rates_by_group.json")
lang = exp1a["language_group3"]
pred_pos = sum(g["TP"] + g["FP"] for g in lang)
pred_n = sum(g["n"] for g in lang)
pred_rate = pred_pos / pred_n

step2, p_step2 = load("diag", "step2_signal_check.json")
d = step2["contra_prob_vs_pharm_label__ambiguous_as_negative_n299"]
pharm_pos, pharm_n = d["n_pos"], d["n"]
pharm_rate = pharm_pos / pharm_n

# Panel (b) is gone. Its four Matthews-correlation rows are Figure 8's
# first family, drawn identically 85 pages later, and Figure 8 is now the
# one place every headline estimate in this thesis lives. What remains is
# the number the rest of the thesis follows: a rate of a whole, so bars.
fig, ax = plt.subplots(figsize=(FIGW, 2.78))
canvas(fig)
# The margins are held at their old ABSOLUTE size and the whole of the
# canvas's growth is given to the axes: the type in the margins is still
# 9pt, so a margin that grew with the canvas would only have bought white
# paper. The top margin is 3pt smaller than it was, which is the space the
# signature bar used to occupy.
fig.subplots_adjust(left=0.2747, right=0.9863, top=0.7306, bottom=0.1559)

# "candidate pool", not "train pool": weak_n is the full weak-labelled
# candidate pool, which is a different and larger denominator than the
# training pool the detector was actually fit on. Chapter 1's caption
# prose says exactly this, and the old label contradicted it.
bars = [("Weak labeller, candidate pool", weak_pos, weak_n, wilson_ci(weak_pos, weak_n)),
        ("Weak labeller, gold set", pred_pos, pred_n, wilson_ci(pred_pos, pred_n)),
        ("Pharmacist truth, gold set", pharm_pos, pharm_n, wilson_ci(pharm_pos, pharm_n))]
ypos = np.arange(len(bars))[::-1]
rates = [b[1] / b[2] for b in bars]
los = [b[3][0] for b in bars]
his = [b[3][1] for b in bars]
errs = [[r - lo for r, lo in zip(rates, los)],
        [hi - r for r, hi in zip(rates, his)]]
# The weak labeller is Llama 3 8B (Chapter 3), so it carries Llama's hue on
# both of its rows; the pharmacist carries the truth hue.
hues = [PLUM, PLUM, EMERALD]

# 1.56, not 1.86. The right third of the old range held nothing at all: the
# bracket and the multiplier end at 1.42 and the ticks stop at 1.0. On a
# canvas that is now 9 percent wider that dead band was the largest single
# area of white on the sheet, and closing it makes the three measured bars
# 19 percent longer without changing one number.
ax.set_xlim(0, 1.56)
ax.set_ylim(-0.66, 2.66)
# The gap between the pharmacist's rate and the weak labeller's on the same
# 299 pairs IS the finding, so it is drawn as a region rather than left for
# the reader to measure off the axis.
band(ax, pharm_pos / pharm_n, pred_pos / pred_n, SCARLET_WASH, zorder=0)
# Flat fills. The 1.0pt DEEP outline fought the cased interval running
# through the bar and made three bars read as three outlined boxes; the
# CORE fill on paper already has all the edge it needs.
ax.barh(ypos, rates, color=[h.CORE for h in hues], height=0.62,
        linewidth=0, edgecolor="none", zorder=3)
# v3 (this round): drawing the interval through the bar is the defect,
# not the stop it was drawn in. Measured WCAG contrast of the foreground
# line against the CORE fill it crossed: hue.DEEP on the plum bar 2.00:1,
# on the emerald bar 2.56:1; INK on the plum bar 4.02:1; PAPER on the plum
# bar 4.42:1. None of the tones this system has clear the 4.5:1 a thin
# line needs to read at this figure's print size (0.98 linewidth) -- the
# bar fill itself is the problem, not the line colour drawn on it. The
# interval is moved clear of the bar instead: a floating I-beam on bare
# paper, directly above each bar, aligned to the same x-range as its own
# estimate (same lo/hi, same rate) so it is read as that bar's
# uncertainty and nothing else's. INK on PAPER is 17.76:1 and no hue can
# swallow it, so this needs no casing -- there is no fill under it to
# case against. _iv_dy clears the bar's own half-height (0.31 data units)
# by about two points at this axis's scale, and stays well inside the
# 0.38-unit gap to the row above, so the cap never touches a bar edge.
_iv_dy = 0.46
for y, r, e_lo, e_hi in zip(ypos, rates, errs[0], errs[1]):
    ax.errorbar([r], [y + _iv_dy], xerr=[[e_lo], [e_hi]], fmt="none",
                ecolor=INK, zorder=5, **ERRBAR)
# One format for every bar on this axis, not one per bar. The x axis
# itself is ticked in decimals (0.00 .. 1.00, labelled "Positive rate"),
# so a bar label in percent form disagrees with the axis it sits on -- a
# reader reads "87.3%" against tick marks reading "0.75, 1.00" and has to
# convert one of them. This used to print the pharmacist's row (index 2)
# as a decimal, matching the 0.0401 fixed in the chapters' prose, and the
# two weak-labeller rows as one-decimal percentages; that is two formats
# on one decimal-ticked axis. All three are now the same four-decimal
# form the pharmacist's rate already had, computed from the same counts
# loaded above (1,425/1,679, 261/299, 12/299), so nothing is hardcoded and
# nothing on this axis needs converting to read the other two.
for i, (y, b, r, hi, hue) in enumerate(zip(ypos, bars, rates, his, hues)):
    label = f"{r:.4f}"
    ax.text(hi + 0.030, y, label, va="center", ha="left",
            fontsize=T_SERIES, color=hue.DEEP, zorder=6)
ax.set_yticks(ypos)
ax.set_yticklabels([f"{b[0]}\n({b[1]:,} of {b[2]:,})" for b in bars],
                   fontsize=T_SERIES, linespacing=1.3)
ax.set_xticks([0, 0.25, 0.5, 0.75, 1.0])
ax.set_xlabel("Positive rate")
frame(ax)

# The claim numeral. The bracket spans only the two bars measured on the
# same 299-pair gold set; the training-pool bar has a different denominator
# and is not part of the comparison.
gold_ratio = (pred_pos / pred_n) / (pharm_pos / pharm_n)
x_br = 1.16
ax.plot([x_br] * 2, [ypos[2], ypos[1]], color=SCARLET_CORE, lw=LW_DATA,
        solid_capstyle="butt", zorder=6)
for y in (ypos[1], ypos[2]):
    ax.plot([x_br, x_br - 0.045], [y, y], color=SCARLET_CORE, lw=LW_DATA,
            zorder=6)
# The multiplier sits beside its bracket; what it is a ratio OF is the
# figure's deck, under the headline, where a subtitle belongs. Wedged into
# the axes it had nowhere to go: beside the bracket it wrapped to four lines
# and the last was struck through by the bottom spine, and moved below the
# bracket it ran across the wash through empty plot space.
ax.text(x_br + 0.06, (ypos[1] + ypos[2]) / 2, f"{gold_ratio:.0f}x",
        va="center", ha="left", fontsize=T_CLAIM, fontweight="bold",
        color=SCARLET_DEEP, zorder=6)

# The bars above are now all four-decimal rates (fix for the mixed-format
# defect: percent labels on a decimal-ticked axis). Restating the weak
# labeller's clause here as "87.3%" would reintroduce exactly that
# mismatch one level up, between the bar and the headline that names it,
# so this line is stated in the raw counts instead -- the same counts
# already printed on the y-axis tick labels below each bar ("261 of
# 299"), which cannot conflict with a rate format because they are not a
# rate. The pharmacist's clause keeps 0.0401, the same four-decimal
# fraction its own bar now carries and the one already fixed by the
# surrounding prose, so nothing here is in two formats.
signature(fig, PLUM_DEEP,
          f"The weak labeller calls {pred_pos} of {pred_n} gold-set "
          f"pairs contradictions.\n"
          f"The pharmacist's base rate is {pharm_pos/pharm_n:.4f}.",
          deck=f"{gold_ratio:.0f}x: the weak labeller's marginal over the "
               f"pharmacist's, on the same {pharm_n} pairs.")
fname = "fig-intro-headline.pdf"
fig.savefig(os.path.join(FIGDIR, fname))
plt.close(fig)
register(fname, "fig:intro-headline",
         [p_step01, p_exp1a, p_step2],
         "The left-panel middle bar is the weak labeller marginal, 261 of 299 "
         "(gold.weaklabel_contradiction); ACCORD predicts contradiction for 278 of 299 "
         "(gold.accord_pred_contradiction, reconciliation_c28.json). The Matthews "
         "correlation panel that used to sit under this one is deleted: those four "
         "estimates are fig:forest's first family.")


## Figure 3: `fig-cohort-flow`


In [9]:
p_icd = os.path.join(RESULTS_ACCORD, "stats", "cohort_icd_counts.json")
hf = json.load(open(p_icd))["heart_failure"]

p_transfer = os.path.join(RESULTS_ACCORD, "generality", "transfer_results_correct.json")
n_candidates = json.load(open(p_transfer))["hf_reference"]["n_candidates"]

p_step2 = os.path.join(RESULTS_ACCORD, "diag", "step2_signal_check.json")
d_step2 = json.load(open(p_step2))
d299 = d_step2["contra_prob_vs_pharm_label__ambiguous_as_negative_n299"]
d253 = d_step2["contra_prob_vs_pharm_label__ambiguous_excluded_n253"]
d_amb = d_step2["contra_prob_vs_pharm_label__ambiguous_as_positive_n299"]
# The ambiguous count is not stored on its own line. It is the difference
# between the two positive counts recorded in the same file on the same 299
# rows, ambiguous counted positive against ambiguous counted negative, which is
# the same subtraction the prevalence figure performs.
n_ambiguous = d_amb["n_pos"] - d299["n_pos"]

# Four stages have no primary artifact inside this repository: their upstream
# summaries live outside it under the data agreement. Their values are read from
# the thesis register, which is committed here and records the artifact each one
# came from.
p_facts = os.path.join(RESULTS_ACCORD, "VERIFIED_FACTS.json")
facts = json.load(open(p_facts))
fact = lambda key: facts[key]["value"]

n_both_roles = fact("cohort_chain.has_both_roles")
n_pairs = fact("cohort_chain.valid_admissions_both_selected")
n_note_rows = fact("cohort_chain.total_note_pair_rows")
n_mentions = fact("extraction.medication_extraction.total_mentions")
n_classified = fact("extraction.medication_extraction.classified_mentions")
n_unclassified = fact("extraction.medication_extraction.unclassified_derived")
n_drawn = fact("extraction.annotation_sample.total_sampled")

print("fig-cohort-flow: values plotted")
print(f"  ICD prefix admissions      {hf['pre_los']:>8,}")
print(f"  removed by length of stay  {hf['removed']:>8,}")
print(f"  after length of stay       {hf['post_los']:>8,}")
print(f"  both note roles present    {n_both_roles:>8,}")
print(f"  note pairs selected        {n_pairs:>8,}   note rows {n_note_rows:,}")
print(f"  medication mentions        {n_mentions:>8,}")
print(f"  no action class            {n_unclassified:>8,}")
print(f"  action classified          {n_classified:>8,}")
print(f"  candidates after prefilter {n_candidates:>8,}")
print(f"  annotated pairs            {d299['n']:>8,}   of {n_drawn:,} drawn")
print(f"  coded ambiguous            {n_ambiguous:>8,}")
print(f"  non-ambiguous core         {d253['n']:>8,}   positives {d253['n_pos']}")


fig-cohort-flow: values plotted
  ICD prefix admissions        13,608
  removed by length of stay        11
  after length of stay         13,597
  both note roles present       9,820
  note pairs selected           9,783   note rows 19,566
  medication mentions         219,683
  no action class             131,347
  action classified            88,336
  candidates after prefilter    1,978
  annotated pairs                 299   of 300 drawn
  coded ambiguous                  46
  non-ambiguous core              253   positives 12


**Step 2. The figure.** Drawn only from the values printed above, then written to `fig-cohort-flow.pdf`.

In [ ]:
# Every count is read from the thesis register by key. The register is
# imported above as _FACTS through fact(), which refuses withdrawn keys.
KEYS = [
    "cohort.heart_failure.pre_los", "cohort.heart_failure.post_los",
    "cohort.heart_failure.removed", "cohort_chain.has_both_roles",
    "cohort_chain.valid_admissions_both_selected",
    "cohort_chain.both_roles_not_selectable",
    "extraction.medication_extraction.total_mentions",
    "extraction.medication_extraction.classified_mentions",
    "extraction.medication_extraction.unclassified_derived",
    "extraction.temporal_prefilter.final_candidates",
    "extraction.annotation_sample.total_sampled",
    "c120.g299.n", "c120.g299.n_positives", "c120.g253.n",
    "c120.g253.n_positives",
]
n_pre = fact("cohort.heart_failure.pre_los")
n_post = fact("cohort.heart_failure.post_los")
n_los_removed = fact("cohort.heart_failure.removed")
assert n_pre - n_post == n_los_removed
n_both_roles = fact("cohort_chain.has_both_roles")
n_pairs = fact("cohort_chain.valid_admissions_both_selected")
n_not_selectable = fact("cohort_chain.both_roles_not_selectable")
assert n_both_roles - n_pairs == n_not_selectable
n_mentions = fact("extraction.medication_extraction.total_mentions")
n_classified = fact("extraction.medication_extraction.classified_mentions")
n_unclassified = fact("extraction.medication_extraction.unclassified_derived")
assert n_mentions - n_classified == n_unclassified
n_candidates = fact("extraction.temporal_prefilter.final_candidates")
n_sampled = fact("extraction.annotation_sample.total_sampled")
n_annot = fact("c120.g299.n")
n_pos = fact("c120.g299.n_positives")
n_core = fact("c120.g253.n")
assert fact("c120.g253.n_positives") == n_pos
# Differences computed here, not typed: drops no file records as its own
# exclusion are drawn as dashed stubs.
n_no_roles = n_post - n_both_roles
n_not_sampled = n_candidates - n_sampled
n_not_annotated = n_sampled - n_annot
n_ambiguous = n_annot - n_core
# Cross-check the ambiguous count against the signal-check artifact, which
# records the positives on the same 299 rows both ways.
step2, p_step2 = load("diag", "step2_signal_check.json")
d_neg = step2["contra_prob_vs_pharm_label__ambiguous_as_negative_n299"]
d_amb = step2["contra_prob_vs_pharm_label__ambiguous_as_positive_n299"]
assert d_amb["n_pos"] - d_neg["n_pos"] == n_ambiguous

W, H = W88, 4.34
fig, ax = plt.subplots(figsize=(W, H))
canvas(fig)
fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
ax.set_xlim(0, W * 72)
ax.set_ylim(0, H * 72)
ax.axis("off")
x_right = W * 72 - 1.5

# (stage name, count with unit, stub below this stage or None)
# a stub is (text, recorded): recorded -> solid, a computed difference -> dashed
stages = [
    ("Heart failure, ICD-9 prefix", f"{n_pre:,} admissions",
     (f"{n_los_removed:,} admissions with no\npositive length of stay", True)),
    ("Length-of-stay filter", f"{n_post:,} admissions",
     (f"{n_no_roles:,} admissions without\nboth note types", False)),
    ("Both note types present", f"{n_both_roles:,} admissions",
     (f"{n_not_selectable:,} admissions, no nursing\nnote before discharge", True)),
    ("One note pair per admission", f"{n_pairs:,} pairs", None),
    ("Medication mentions extracted", f"{n_mentions:,} mentions",
     (f"{n_unclassified:,} mentions with\nno action class", True)),
    ("Mentions with an action class", f"{n_classified:,} mentions", None),
    ("After the temporal prefilter", f"{n_candidates:,} candidates",
     (f"{n_sampled:,} sampled from {n_candidates:,}", False)),
    ("Sampled for annotation", f"{n_sampled:,} pairs",
     (f"{n_annot:,} annotated of\n{n_sampled:,} sampled", False)),
    ("Annotated by the pharmacist", f"{n_annot:,} pairs",
     (f"{n_ambiguous:,} pairs coded ambiguous", True)),
    ("Non-ambiguous evaluation core",
     f"{n_core:,} pairs, {n_pos:,} confirmed contradictions", None),
]
assert n_not_sampled > 0 and n_not_annotated >= 0

box_x, box_w = 26.0, 222.0
x_gut = 262.0
box_h, last_h, gap = 17.0, 28.0, 10.0
y_top = H * 72 - 34.0
tops, y = [], y_top
for i in range(len(stages)):
    h = last_h if i == len(stages) - 1 else box_h
    tops.append((y, h))
    y -= h + gap

# Phase groups share one ground tone; the phase is carried by its name,
# the box hue (cobalt extraction, emerald the pharmacist) and the rail.
phases = [
    (range(0, 4), HAIR, MIST, SLATE, SLATE, "cohort construction"),
    (range(4, 7), HAIR, COBALT_TINT, COBALT_DEEP, COBALT_DEEP,
     "extraction and mining"),
    (range(7, 10), HAIR, EMERALD_TINT, EMERALD_DEEP, EMERALD_DEEP,
     "annotation"),
]
phase_of = {}
for rows, wash, tint, deep, rail, name in phases:
    idx = list(rows)
    for i in idx:
        phase_of[i] = (tint, deep)
    b_hi = tops[idx[0]][0] + 5.0
    b_lo = tops[idx[-1]][0] - tops[idx[-1]][1] - 5.0
    ax.add_patch(mpatches.Rectangle((0, b_lo), W * 72, b_hi - b_lo,
                                    facecolor=wash, edgecolor="none",
                                    linewidth=0, zorder=0))
    ax.add_patch(mpatches.Rectangle((7.0, b_lo + 2.0), 3.0,
                                    b_hi - b_lo - 4.0, facecolor=rail,
                                    edgecolor="none", linewidth=0, zorder=1))
    ax.text(17.0, (b_lo + b_hi) / 2, name, ha="center", va="center",
            rotation=90, fontsize=T_NOTE, color=MUTE, zorder=2)

x_stem = box_x + box_w / 2
for i, ((t, h), (name, count, stub)) in enumerate(zip(tops, stages)):
    last = i == len(stages) - 1
    mid = t - h / 2
    if last:
        # The evaluated core: the pharmacist's set, a solid emerald chip.
        chip(ax, box_x, t - h, box_w, h, EMERALD,
             f"{name}\n{count}", fontsize=T_SERIES, radius=3.0, zorder=4)
    else:
        tint, deep = phase_of[i]
        node(ax, box_x, t - h, box_w, h, tint, deep, radius=2.5, zorder=4)
        ax.text(box_x + 7.0, mid, name, ha="left", va="center",
                fontsize=T_SERIES, color=INK, zorder=5)
        # SLATE is too light for type (3.48:1), so the cohort phase's
        # counts are set in GRAPHITE.
        ax.text(box_x + box_w - 7.0, mid, count, ha="right", va="center",
                fontsize=T_SERIES, color=GRAPHITE if deep == SLATE else deep,
                zorder=5)
        arrow(ax, (x_stem, t - h), (x_stem, t - h - gap), color=GRAPHITE,
              lw=LW_CONNECTOR, zorder=6)
    if stub is not None:
        text, recorded = stub
        ls = "solid" if recorded else (0, (3.0, 2.0))
        x_elbow = box_x + box_w + 5.0
        y_gap = t - h - gap / 2
        ax.plot([box_x + box_w, x_elbow], [mid, mid], color=SLATE,
                lw=1.0, zorder=6, solid_capstyle="butt", linestyle=ls)
        ax.plot([x_elbow, x_elbow], [mid, y_gap], color=SLATE,
                lw=1.0, zorder=6, solid_capstyle="butt", linestyle=ls)
        ax.plot([x_elbow, x_gut], [y_gap, y_gap], color=SLATE,
                lw=1.0, zorder=6, solid_capstyle="butt", linestyle=ls)
        lines = text.count("\n") + 1
        ch = 10.0 * lines + 5.0
        p = node(ax, x_gut, y_gap - ch / 2, x_right - 2.0 - x_gut, ch,
                 PAPER if not recorded else MIST, SLATE, radius=2.0,
                 zorder=4, lw=1.0)
        if not recorded:
            p.set_linestyle((0, (3.0, 2.0)))
        ax.text(x_gut + 5.0, y_gap, text, ha="left", va="center",
                fontsize=T_NOTE, color=MUTE if not recorded else GRAPHITE,
                linespacing=1.15, zorder=5)

ax.text(x_gut, y_top + 6.0, "left the path", ha="left", va="bottom",
        fontsize=T_NOTE, color=MUTE, style="italic")

signature(fig, EMERALD_DEEP,
          f"Of {n_candidates:,} candidates, {n_pos:,} contradictions survive "
          f"sampling and annotation.")
fname = "fig-cohort-flow.pdf"
fig.savefig(os.path.join(FIGDIR, fname))
plt.close(fig)
srcs = [_FACTS_PATH] + sorted({fact_source(k) for k in KEYS} - {_FACTS_PATH}) + [p_step2]
register(fname, "fig:cohort-flow", srcs,
         "Path from the source database to the evaluated core, ten stages. Every "
         "count is read from results/accord/VERIFIED_FACTS.json by key ("
         + ", ".join(KEYS) + "), and every right-hand count carries its unit. "
         "Solid stubs are exclusions the register records as counts (11 with no "
         "positive length of stay, 37 with no nursing note before discharge, "
         "131,347 unclassified mentions, 46 coded ambiguous). Dashed stubs are drops "
         "no file records as an exclusion, drawn as differences computed in code "
         "from two register counts: admissions after the length-of-stay filter "
         "minus admissions with both note types; candidates minus the sample; the "
         "sample minus the annotated pairs. The ambiguous count, 299 minus 253, is "
         "cross-checked against the signal-check file. The core is drawn in the "
         "pharmacist's emerald rather than scarlet, which the set reserves for a "
         "failed check. Headline built from the register: of 1,978 candidates, 12 "
         "contradictions survive sampling and annotation.")


## Figure 4: `fig-exemplar`


In [11]:
p_exemplar = os.path.join(RESULTS_ACCORD, "exemplar", "exemplar_correctness_recomputed.json")
d_exemplar = json.load(open(p_exemplar))
ex = d_exemplar["rule_1_2_pooled"]
defect_rate = ex["n_wrong"] / ex["n_resolved"]

print("fig-exemplar: values plotted")
print(f"  rule_1_2_pooled: n_wrong={ex['n_wrong']}, n_resolved={ex['n_resolved']}, defect_rate={defect_rate:.4f}")

# Widened and re-anchored. Previously the trigger caption ran into the
# "Discharge summary" heading, both leader arrows crossed a slot box and struck
# through its label, the right-hand annotations ran off the axes, and the summary
# text was wider than the box drawn around it.

fig-exemplar: values plotted
  rule_1_2_pooled: n_wrong=65, n_resolved=137, defect_rate=0.4745


**Step 2. The figure.** Drawn only from the values printed above, then written to `fig-exemplar.pdf`.

In [ ]:
exemplar, p_exemplar = load("exemplar", "exemplar_correctness_recomputed.json")

# Abstract slots. No synthetic clinical sentences: a thesis about
# instruments that mislead, built on a MIMIC-III data agreement, does not
# put invented clinical text on a canvas where a reader could take it for a
# record. One data unit is one typographic point. No plate.
W, H = W88, 2.76
fig, ax = plt.subplots(figsize=(W, H))
canvas(fig)
fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
ax.set_xlim(0, W * 72)
ax.set_ylim(0, H * 72)
ax.axis("off")

x_brace, box_w = 72.0, 84.0
x_nurse = x_brace + 9.0 + box_w / 2
x_disch = x_nurse + box_w + 7.0
x_note = x_disch + box_w / 2 + 15.0
box_h, pitch = 22.0, 28.5
y_row0 = 133.0
rows_y = [y_row0 - i * pitch for i in range(5)]
x_right = W * 72

# Row 5 is where the rule fired: a wash ground across the whole row.
ax.add_patch(mpatches.Rectangle(
    (0, rows_y[4] - box_h / 2 - 4.5), x_right, box_h + 9.0,
    facecolor=SCARLET_WASH, edgecolor="none", linewidth=0, zorder=0))

# A vertical brace left of M1 to M5: the rule quantifies over the whole set,
# which is the fact the positional selector ignores.
y_top, y_bot = rows_y[0] + box_h / 2, rows_y[4] - box_h / 2
y_mid = (y_top + y_bot) / 2
bx = x_brace
ax.plot([bx + 4.0, bx, bx, bx - 4.0, bx, bx, bx + 4.0],
        [y_top, y_top - 4.0, y_mid + 4.0, y_mid, y_mid - 4.0, y_bot + 4.0,
         y_bot], color=INK, lw=1.2, solid_joinstyle="round", zorder=4)
ax.text(bx - 8.0, y_mid, "the rule\nquantifies\nover the\nwhole set",
        ha="right", va="center", fontsize=T_SERIES, color=INK,
        linespacing=1.2, zorder=5)

for x, head in ((x_nurse, "Nursing note"), (x_disch, "Discharge summary")):
    ax.text(x, y_top + 10.0, head, ha="center", va="bottom", fontsize=10.0,
            color=INK, zorder=5)

nurse_tags = ["M1", "M2", "M3", "M4", "M5"]
disch_tags = ["continuation"] * 4 + ["cessation"]
for i, y in enumerate(rows_y):
    # Nursing-note slots are neutral MIST: cobalt means extraction elsewhere.
    node(ax, x_nurse - box_w / 2, y - box_h / 2, box_w, box_h,
         MIST, SLATE, radius=2.5, zorder=3, lw=1.0)
    ax.text(x_nurse, y, nurse_tags[i], ha="center", va="center",
            fontsize=T_SERIES, color=INK, zorder=5)
    if i < 4:
        node(ax, x_disch - box_w / 2, y - box_h / 2, box_w, box_h,
             MIST, SLATE, radius=2.5, zorder=3, lw=1.0)
        ax.text(x_disch, y, disch_tags[i], ha="center", va="center",
                fontsize=T_SERIES, color=INK, fontweight="bold", zorder=5)
    else:
        chip(ax, x_disch - box_w / 2, y - box_h / 2, box_w, box_h,
             SCARLET, disch_tags[i], fontsize=T_SERIES, radius=2.5,
             zorder=3)

# Row 1 is the exemplar everyone was shown: outlined as a row.
pad = 3.0
ax.add_patch(mpatches.Rectangle(
    (x_nurse - box_w / 2 - pad, rows_y[0] - box_h / 2 - pad),
    (x_disch + box_w / 2) - (x_nurse - box_w / 2) + 2 * pad, box_h + 2 * pad,
    facecolor="none", edgecolor=INK, linewidth=2.0, zorder=6))

notes = [
    (0, "shown", "selected by position; what\nthe annotator, the weak\n"
                 "labeller and the model saw", MUTE),
    (4, "fired", "the mention the rule\nmatched, dose adjacent", MUTE),
]
for i, tag, text, tone in notes:
    y = rows_y[i]
    x0 = x_disch + box_w / 2 + (pad + 2.0 if i == 0 else 3.0)
    ax.plot([x0, x_note - 4.0], [y, y], color=SLATE, lw=1.0,
            solid_capstyle="butt", zorder=4)
    ax.text(x_note, y, tag, ha="left", va="center", fontsize=T_SERIES,
            color=INK, fontweight="bold", zorder=5)
    # Row 1's note sits under its tag; row 5's sits above its tag, on
    # paper beside row 4, so it stays inside the canvas and off the wash.
    if i == 0:
        ax.text(x_note, y - 7.0, text, ha="left", va="top",
                fontsize=T_NOTE, color=tone, linespacing=1.2, zorder=5)
    else:
        ax.text(x_note, y + 18.0, text, ha="left", va="bottom",
                fontsize=T_NOTE, color=tone, linespacing=1.2, zorder=5)
    assert x_note + 104.0 < x_right, "note leaves the canvas"

ex_lo = 1.0 - fact("exemplar.rule_1_2_pooled_exemplar_correct_rate_in_299.rate")
ex_mid = fact("exemplar_recomputed.rule_1_2_pooled.wrong_rate")
ex_hi = fact("exemplardirect.defect_rate")
signature(fig, SCARLET_DEEP,
          "The rule fires on a set; the exemplar is picked by position.",
          deck=(f"One schematic candidate; miss rate {ex_lo:.0f}, "
                f"{ex_mid:.4f} or {ex_hi:.4f} by instrument."))
fname = "fig-exemplar.pdf"
fig.savefig(os.path.join(FIGDIR, fname))
plt.close(fig)
register(fname, "fig:exemplar",
         [_FACTS_PATH, p_exemplar,
          fact_source("exemplar.rule_1_2_pooled_exemplar_correct_rate_in_299.rate"),
          fact_source("exemplardirect.defect_rate")],
         "Schematic; slots are abstract labels (M1 to M5, action classes written out as "
         "continuation and cessation) and carry no clinical text. Row 1, the exemplar "
         "selected by position, carries a 2.0 pt INK outline and a 'shown' tag; row 5, "
         "where the rule matched, keeps its SCARLET_WASH ground and a 'fired' tag. A "
         "vertical brace left of M1 to M5 states that the rule quantifies over the whole "
         "set. Nursing-note slots are neutral MIST (cobalt means extraction elsewhere). "
         "No plate. The one-line deck prints the three instruments' miss rates, read "
         "from the register by key: exemplar.rule_1_2_pooled_exemplar_correct_rate_in_299.rate "
         "(one minus it, 0), exemplar_recomputed.rule_1_2_pooled.wrong_rate (0.4745), "
         "exemplardirect.defect_rate (0.9315). Gate note: tools/measure_ink.py holds this "
         "canvas to chroma 0.055; it measures 0.023, because the cobalt slot fills and "
         "strokes the floor was set against are removed by the critique and the only "
         "saturated element left is the cessation chip. The floor was not met by adding "
         "colour; it needs lowering by the gate owner.")


## Figure 5: `fig-prevalence-response`


In [13]:
p_sweep = os.path.join(RESULTS_ACCORD, "prevalence_replication", "sweep_results.json")
d_sweep = json.load(open(p_sweep))
models = d_sweep["models"]
prompts = ["A_bare", "B_baserate_abstain"]
# The second row label is wrapped: on one line it ran the full height of its
# panel and collided with the row above it.
prompt_label = {"A_bare": "bare prompt", "B_baserate_abstain": "base rate stated,\nabstention allowed"}

# The "clinical" comparison point: same weak-label-vs-pharmacist shift as figures 1 and 2,
# recomputed here from the same two diagnostic files (not typed in).
p_step01 = os.path.join(RESULTS_ACCORD, "diag", "step0_1_join_and_calibration.json")
d_step01 = json.load(open(p_step01))
xtab = d_step01["weak_confidence_is_degenerate"]["crosstab_label_x_confidence"]
weak_pos, weak_neg = xtab["0.95"]["contradiction"], xtab["1.0"]["not_contradiction"]
clinical_y = weak_pos / (weak_pos + weak_neg)

p_step2 = os.path.join(RESULTS_ACCORD, "diag", "step2_signal_check.json")
d_step2 = json.load(open(p_step2))
clinical_x = d_step2["contra_prob_vs_pharm_label__ambiguous_as_negative_n299"]["base_rate"]

print("fig-prevalence-response: values plotted")
print(f"  models: {models}")
print(f"  clinical marker (true prevalence, labeller positive rate): ({clinical_x:.4f}, {clinical_y:.4f})")
for pr in prompts:
    for m in models:
        pts = {p: d_sweep["results"][m][pr].get(p) for p in ["0.02", "0.05", "0.1", "0.25"]}
        rates = {p: (e["positive_rate_of_decided"] if e else None) for p, e in pts.items()}
        print(f"  {m:14s} / {prompt_label[pr]:38s}: {rates}")

fig-prevalence-response: values plotted
  models: ['llama3:8b', 'granite3.3:8b', 'qwen3:8b']
  clinical marker (true prevalence, labeller positive rate): (0.0401, 0.8487)
  llama3:8b      / bare prompt                           : {'0.02': 0.07, '0.05': 0.17, '0.1': 0.06, '0.25': 0.12}
  granite3.3:8b  / bare prompt                           : {'0.02': 0.05, '0.05': 0.07, '0.1': 0.1, '0.25': 0.16}
  qwen3:8b       / bare prompt                           : {'0.02': 0.02, '0.05': 0.03, '0.1': 0.05, '0.25': 0.09}
  llama3:8b      / base rate stated,
abstention allowed  : {'0.02': 0.0410958904109589, '0.05': 0.10975609756097561, '0.1': 0.05405405405405406, '0.25': 0.07352941176470588}
  granite3.3:8b  / base rate stated,
abstention allowed  : {'0.02': 0.030612244897959183, '0.05': 0.04, '0.1': 0.09, '0.25': 0.17}
  qwen3:8b       / base rate stated,
abstention allowed  : {'0.02': 0.0, '0.05': 0.0, '0.1': 0.025, '0.25': 0.02666666666666667}


**Step 2. The figure.** Drawn only from the values printed above, then written to `fig-prevalence-response.pdf`.

In [ ]:
# =============================================================================
# Fig 5 -- fig-prevalence-response
# =============================================================================
# THIS FIGURE WAS REBUILT AROUND ITS OWN MOST IMPORTANT DATUM. It used to be a
# two-by-three grid of small multiples, one panel per model per prompt, on a y
# axis that stopped at 0.38 -- and the clinical operating point, the one place
# on the canvas where the deployed labeller meets the real base rate, did not
# fit on that axis. It was drawn as an arrow leaving the top of the first panel
# with its coordinates written beside it. An annotation that names a value is
# not a plot of it: the reader could not see how far above every swept point
# the real one sits, which is the whole reason the sweep understates the
# problem.
#
# The axis now reaches it, so it is a mark on the canvas like any other, and
# the cost is stated rather than hidden: the swept points occupy the bottom
# quarter of each panel instead of most of it, and the calibration diagonal is
# shallow rather than at forty-five degrees. That trade is the correct one.
# The claim is not how far apart 0.07 and 0.17 are, which is a table; the claim
# is that every response is flat against prevalence and that all of them sit an
# order of magnitude below where the deployed labeller actually operates.
#
# Six panels became two, one per prompt, three models each, so a reader
# compares models by looking at one panel rather than by saccading between
# three.
def make_fig_prevalence_response():
    # The copy under results/, not the one under accord/prevalence_replication/.
    # The two files are byte-identical; only this one is inside the tree that gets
    # copied to a runtime, so reading it is what lets this figure build in Colab.
    d = json.loads((ROOT / "results/accord/prevalence_replication/sweep_results.json").read_text())
    models, prompts = d["models"], ["A_bare", "B_baserate_abstain"]
    # ROUND 7: the clinical operating point is no longer drawn. It sat on axes whose
    # x is SNLI prevalence, a different pool, prompt and kind of negative from the
    # clinical pool, which invited exactly the comparison the chapter says the
    # design cannot license. The clinical over-calling range is named in the deck
    # instead, read from the register, and is not given a coordinate.
    clin_lo = fact("overcall.qwen3_8b.pool.ratio")
    clin_hi = fact("overcall.llama3_8b.pool.ratio")
    cells = [(m, pr, lv, e["over_call_ratio"])
             for m in models for pr in prompts
             for lv, e in d["results"][m][pr].items()]
    n_cells = len(cells)
    n_under = sum(1 for c in cells if c[3] < 1)
    top = max(cells, key=lambda c: c[3])
    # The maximum is the register's own mirror of that cell, not only the artifact.
    assert abs(fact(f"publicsweep.results.{top[0]}.{top[1]}.{top[2]}.over_call_ratio")
               - top[3]) < 1e-9
    assert max(c[3] for c in cells) < clin_lo

    PL = {"A_bare": "bare prompt",
          "B_baserate_abstain": "base rate stated, abstention allowed"}
    # The three public models carry no hue: INK, MUTE and SLATE, told apart by
    # marker and dash pattern (style.PUBLIC_MODEL_MARK), so hue stays reserved
    # for the deployed labeller, the ensemble and failed checks. MODEL_HUE is
    # the name check_figure_invariants.py compares against style.ENTITY_HUE.
    MODEL_HUE = {"llama3:8b": INK, "granite3.3:8b": MUTE,
                 "qwen3:8b": SLATE}
    # Series named once, in the left panel, at their own mark.
    LABEL_AT = {
        "A_bare": {"llama3:8b": ("0.05", +1), "granite3.3:8b": ("0.1", +1),
                   "qwen3:8b": ("0.25", -1)},
        "B_baserate_abstain": {},
    }
    # Models are dodged on x so their intervals do not print on one another.
    DODGE = {"llama3:8b": -0.004, "granite3.3:8b": 0.0, "qwen3:8b": +0.004}
    Y_LO, Y_HI, X_HI = -0.012, 0.32, 0.27
    SWEEP_P = ["0.02", "0.05", "0.1", "0.25"]

    fig, axes = plt.subplots(1, 2, figsize=(FIGW, 3.30), sharey=True)
    canvas(fig)
    fig.subplots_adjust(left=0.0930, right=0.9830, top=0.8050, bottom=0.1260,
                        wspace=0.075)
    for c, pr in enumerate(prompts):
        ax = axes[c]
        gx = np.linspace(0, X_HI, 2)
        ax.fill_between(gx, gx, Y_HI, facecolor=FOG, edgecolor="none",
                        linewidth=0, zorder=0)
        ax.plot([0, X_HI], [0, X_HI], color=SLATE, linestyle=REF_DASH,
                linewidth=LW_REF, zorder=2)
        for m in models:
            xs, ys, lo, hi = [], [], [], []
            for p in SWEEP_P:
                e = d["results"][m][pr].get(p)
                if not e:
                    continue
                xs.append(float(p) + DODGE[m]); ys.append(e["positive_rate_of_decided"])
                a, b = wilson_ci(e["n_contradiction"], e["n_decided"])
                lo.append(a); hi.append(b)
            marker, ls, tone = PUBLIC_MODEL_MARK[m]
            assert tone == MODEL_HUE[m]
            ax.errorbar(xs, ys, yerr=[[y - a for y, a in zip(ys, lo)],
                                      [b - y for y, b in zip(ys, hi)]],
                        fmt="none", ecolor=tone, elinewidth=1.1,
                        capthick=1.1, capsize=2.6, zorder=3)
            ax.plot(xs, ys, color=tone, linestyle=ls, lw=1.8, zorder=4,
                    marker=marker, ms=6.0, markerfacecolor=tone,
                    markeredgecolor=PAPER, markeredgewidth=0.8)
            if m in LABEL_AT[pr]:
                at, side = LABEL_AT[pr][m]
                i = SWEEP_P.index(at)
                y_anchor = (hi[i] + 0.008) if side > 0 else (lo[i] - 0.006)
                x_anchor = xs[i] if side > 0 else X_HI - 0.006
                case(ax.text(x_anchor, y_anchor, m,
                             ha="center" if side > 0 else "right",
                             va="bottom" if side > 0 else "top",
                             fontsize=T_SERIES, color=INK, zorder=6))
        if c == 0:
            case(ax.text(0.012, Y_HI - 0.010, "over-prediction",
                         ha="left", va="top", fontsize=T_NOTE, color=MUTE,
                         zorder=5))
        ax.set_xlim(0, X_HI); ax.set_ylim(Y_LO, Y_HI)
        # The diagonal is labelled in both panels, at its angle on the page.
        fig.canvas.draw()
        bb = ax.get_window_extent()
        ang = math.degrees(math.atan2(bb.height / (Y_HI - Y_LO),
                                      bb.width / X_HI))
        case(ax.text(0.185, 0.197, "perfect calibration", fontsize=T_NOTE,
                     color=MUTE, ha="center", va="bottom", rotation=ang,
                     rotation_mode="anchor", zorder=5))
        ax.set_xticks([0, 0.1, 0.2])
        ax.set_yticks([0, 0.1, 0.2, 0.3])
        ax.set_title(PL[pr], fontsize=T_SERIES, pad=5, color=INK)
        ax.tick_params(labelsize=T_TICK)
        frame(ax, grid="y")
    fig.supxlabel("True prevalence", fontsize=T_AXIS, y=0.016)
    axes[0].set_ylabel("Labeller predicted-positive rate", fontsize=T_AXIS)
    signature(fig, INK,
              f"On public text, {n_under} of {n_cells} labeller settings under-call,\n"
              f"and the largest over-call is {top[3]:.1f} times.")
    fig.savefig(HERE / "fig-prevalence-response.pdf"); plt.close(fig)
    print("wrote fig-prevalence-response.pdf")


## Figure 6: `fig-falsification`


In [15]:
p_agg = os.path.join(RESULTS_ACCORD, "decontam", "accord_v2_decontam_gold_aggregate.json")
d_agg = json.load(open(p_agg))
aN, aD = d_agg["aggregate"]["alpha_n"]["mean"], d_agg["aggregate"]["alpha_d"]["mean"]
sN, sD = d_agg["aggregate"]["alpha_n"]["std"], d_agg["aggregate"]["alpha_d"]["std"]

p_transfer = os.path.join(RESULTS_ACCORD, "generality", "transfer_results_correct.json")
d_transfer = json.load(open(p_transfer))
f1 = [d_transfer["hf_reference"]["rule_type_f1"], d_transfer["sepsis"]["rule_type_f1"], d_transfer["aki"]["rule_type_f1"]]
cohort_labels = ["HF\n(in domain)", "sepsis", "AKI"]
n_cand = [d_transfer["hf_reference"]["n_candidates"], d_transfer["sepsis"]["n_candidates"],
          d_transfer["aki"]["n_candidates"]]
# The three seeds behind each mean, read from the same file the means come from.
seeds = sorted(d_agg["per_seed"])
seed_N = [d_agg["per_seed"][s]["authority_scale"]["alpha_nursing"] for s in seeds]
seed_D = [d_agg["per_seed"][s]["authority_scale"]["alpha_discharge"] for s in seeds]

p_audit = os.path.join(RESULTS_ACCORD, "number_provenance_audit.json")
d_audit = json.load(open(p_audit))
claim = next(r for r in d_audit["records"] if "45.2" in r["claim"])
# The 45.2% held-out rule-accuracy reference line has been REMOVED from this
# figure: 0.4515 appears in no result file anywhere on disk, only in figure code
# and in the audit record printed below, which flags it as untraceable.

print("fig-falsification: values plotted")
print(f"  alpha_N (nursing authority scale), 3-seed mean +/- std : {aN:.4f} +/- {sN:.4f}")
print(f"  alpha_D (discharge authority scale), 3-seed mean +/- std: {aD:.4f} +/- {sD:.4f}")
for lab, v, nc in zip(cohort_labels, f1, n_cand):
    print(f"  rule-type F1, {lab.replace(chr(10), ' '):16s}: {v:.4f}  (n={nc})")
print(f"  alpha_N per seed: {['%.4f' % v for v in seed_N]}")
print(f"  alpha_D per seed: {['%.4f' % v for v in seed_D]}")
print("  alpha initialisation 0.5 / 1.0: a stated design constant, not a measurement;")
print("    it is in no result file, and the figure legend now says so.")
print("  NOT plotted (removed): held-out rule-prediction accuracy 0.4515")
print(f"  number_provenance_audit.json claim record: {claim}")

fig-falsification: values plotted
  alpha_N (nursing authority scale), 3-seed mean +/- std : 0.5076 +/- 0.0006
  alpha_D (discharge authority scale), 3-seed mean +/- std: 1.0038 +/- 0.0010
  rule-type F1, HF (in domain)  : 0.9603  (n=1978)
  rule-type F1, sepsis          : 0.0654  (n=775)
  rule-type F1, AKI             : 0.0595  (n=1983)
  alpha_N per seed: ['0.5070', '0.5074', '0.5084']
  alpha_D per seed: ['1.0024', '1.0042', '1.0047']
  alpha initialisation 0.5 / 1.0: a stated design constant, not a measurement;
    it is in no result file, and the figure legend now says so.
  NOT plotted (removed): held-out rule-prediction accuracy 0.4515
  number_provenance_audit.json claim record: {'claim': '45.2% held-out rule-prediction accuracy (recomputes to 0.4515)', 'tex_location': 'ch6:291', 'artifact': 'gold-standard CSV per ch6 caption', 'artifact_exists': False, 'reported': 0.452, 'recomputed': None, 'verdict': 'UNTRACEABLE', 'blast_radius': 'body', 'note': "Caption asserts it 'recompu

**Step 2. The figure.** Drawn only from the values printed above, then written to `fig-falsification.pdf`.

In [ ]:
# ---- fig:falsification -----------------------------------------------------
agg_all = json.loads((SUPP / "accord_v2_decontam_gold_aggregate.json").read_text())
agg = agg_all["aggregate"]
aN, aD = agg["alpha_n"]["mean"], agg["alpha_d"]["mean"]
seeds = sorted(agg_all["per_seed"])
seed_N = [agg_all["per_seed"][s_]["authority_scale"]["alpha_nursing"] for s_ in seeds]
seed_D = [agg_all["per_seed"][s_]["authority_scale"]["alpha_discharge"] for s_ in seeds]
INIT_N, INIT_D = 0.5, 1.0          # design constants, CORRECTIONS.md C1

# The transfer F1 panel that used to sit here (a bar chart of rule-head F1
# across three cohorts, drawn from zero with no interval) moved to a table the
# author builds in the .tex, so this script no longer reads
# transfer_results_correct.json. This is now a single-panel figure: one
# canvas, one claim, no gutter to widen or narrow.
fig, a = plt.subplots(figsize=(FIGW, 2.35))
canvas(fig)
# Height dropped from 2.68in to 2.35in: the removed panel's own width, not its
# height, funded the freed space, so the cut here is the vertical budget a
# two-line "(a)...(b)..." headline and a panel-letter label no longer need
# (one declarative headline sits at 0.19in of text against the old 0.4in of
# two lines, and there is no second panel_label() call to clear at all). The
# axes keep their former height (top-bottom fraction * 2.35in = 1.35in,
# against 1.41in before): the dumbbell's own row spacing did not shrink. What
# widened is the axes: with no second panel and no wspace gutter to share the
# canvas with, this panel now runs left=0.1172 (unchanged absolute margin,
# same width class) to right=0.9850, versus roughly half that width before.
fig.subplots_adjust(left=0.1172, right=0.9850, top=0.8181, bottom=0.2450)

# ---- the dumbbell -----------------------------------------------------------
# WHAT CAME OFF THIS PANEL, and why. The author called this figure busy. This
# panel carried a tinted band, a two-line band label, two dumbbells, three seed
# dots per row, a percentage at each dumbbell's head and a two-line row label:
# six kinds of thing on two rows of data. Two of them said the same sentence
# twice. The band label read "neither left a 2% band" and the row labels read
# "+0.4%" and "+1.5%", which is the same claim in prose and in figures, and the
# figures are the measurement. THE PROSE WENT AND THE NUMBERS STAYED. The band
# stays as a ground and is named on the axis instead, where a tolerance belongs;
# the seed dots stay, because three seeds landing on top of each other is
# evidence and not decoration.
TOL = 0.02
# WASH, not TINT. The band covers three quarters of this panel, and at TINT
# that is a saturated field with two grey dumbbells on it: the ground was
# louder than the data it exists to locate. It costs nothing measurable --
# SCARLET_TINT sits at HSV saturation 0.233, under the 0.25 gate the ink tool
# counts at, so it registered as chroma 0 either way.
a.axvspan(1 - TOL, 1 + TOL, facecolor=SCARLET_WASH, edgecolor="none",
          linewidth=0, zorder=0)
moved = []
for i, (init, mean, sds) in enumerate(((INIT_N, aN, seed_N),
                                       (INIT_D, aD, seed_D))):
    ratio = mean / init
    moved.append(abs(ratio - 1))
    a.plot([1.0, ratio], [i, i], color=GRAPHITE, lw=2.4,
           solid_capstyle="butt", zorder=3)
    # The three seeds spread 10.8pt about a 6.6pt marker, so on the dumbbell's
    # own line they were neither hidden nor visible: a grey crescent behind the
    # final dot that reads as a print smudge. They drop to their own sub-row,
    # true x, where three dots read as three seeds.
    a.plot([s_ / init for s_ in sds], [i - 0.30] * len(sds), "o", color=SLATE,
           ms=MS_SEED, markeredgewidth=0, zorder=5)
    a.plot([1.0], [i], "o", markerfacecolor=PAPER, markeredgecolor=GRAPHITE,
           markeredgewidth=1.2, ms=MS_DOT, zorder=4)
    a.plot([ratio], [i], "o", color=GRAPHITE, ms=MS_DOT, zorder=6)
    # GRAPHITE at 9pt: the movement is a measured quantity that is not an
    # entity, the dumbbell it labels is drawn in GRAPHITE, and a series is
    # labelled in its own mark's tone. It was 8.5pt MUTE, which is the tone for
    # secondary annotation, and this is the panel's primary measurement.
    a.text(ratio + 0.0030, i, f"{(ratio - 1) * 100:+.1f}%",
           fontsize=T_SERIES, color=GRAPHITE, va="center", ha="left",
           zorder=6)
a.set_yticks([1, 0])
a.set_yticklabels([r"$\alpha_D$" + "\n(set to 1.00)",
                   r"$\alpha_N$" + "\n(set to 0.50)"],
                  fontsize=T_SERIES, linespacing=1.25)
a.set_ylim(-0.60, 1.45)
a.set_xlim(0.9755, 1.0295)
a.set_xticks([0.98, 0.99, 1.00, 1.01, 1.02])
a.set_xticklabels(["0.98", "0.99", "1.00", "1.01", "1.02"])
# The band is named on the axis, not on the field. A tolerance is a property
# of the scale, and the two tick labels the band runs between already read
# 0.98 and 1.02, so this line confirms what the axis shows instead of asserting
# a second time what the two percentages beside the dumbbells already measure.
a.set_xlabel("Final value, as a multiple of the initialisation\n"
             f"(shaded band: within {TOL * 100:.0f}% of it)",
             linespacing=1.25)
a.tick_params(axis="y", length=0)
frame(a)

signature(fig, SCARLET_DEEP,
          "Two parameters reported as learned did not move.")
fig.savefig(HERE / "fig-falsification.pdf"); plt.close(fig)
man["fig:falsification"] = {"file": "fig-falsification.pdf",
    "source": "accord_supp/results/accord_v2_decontam_gold_aggregate.json",
    "values": {"alpha_n": aN, "alpha_d": aD, "alpha_n_per_seed": seed_N,
               "alpha_d_per_seed": seed_D,
               "alpha_init_stated_design_constant": [INIT_N, INIT_D],
               "alpha_n_ratio": aN / INIT_N, "alpha_d_ratio": aD / INIT_D},
    "removed": "the rule-head transfer F1 panel (moved to a table in the .tex; "
               "see results/accord/generality/transfer_results_correct.json) "
               "and the 45.2% held-out rule-accuracy reference line dropped in "
               "an earlier pass (0.4515 is in no result file)"}


## Figure 7: `fig-power`


In [17]:
p_power = os.path.join(RESULTS_ACCORD, "section6_auroc_power.json")
pw = json.load(open(p_power))
mda = pw["minimum_detectable_auc"]
n_pos_study = pw["n_positives"]
n_neg_study = pw["n_negatives"]
alpha_study = pw["alpha"]
power_study = pw["power"]

p_task_runs = os.path.join(RESULTS, "task_runs.json")
d_task_runs = json.load(open(p_task_runs))
mort_lineage = d_task_runs["concat_s1"]["data_lineage"]
mort_n_test = mort_lineage["test_n"]
mort_n_pos = mort_lineage["test_n_pos"]
mort_n_neg = mort_n_test - mort_n_pos

def hanley_mde(n_pos, n_neg, alpha=alpha_study, power=power_study):
    """Smallest AUROC distinguishable from 0.5 at the given power, Hanley-McNeil variance,
    using both class counts."""
    z = norm.ppf(1 - alpha / 2) + norm.ppf(power)
    out = []
    for m, n in zip(np.atleast_1d(n_pos), np.atleast_1d(n_neg)):
        lo, hi = 0.5, 0.9999
        for _ in range(80):
            a = (lo + hi) / 2
            q1, q2 = a / (2 - a), 2 * a * a / (1 + a)
            v = (a * (1 - a) + (m - 1) * (q1 - a * a) + (n - 1) * (q2 - a * a)) / (m * n)
            if (a - 0.5) / math.sqrt(max(v, 1e-12)) < z:
                lo = a
            else:
                hi = a
        out.append((lo + hi) / 2)
    return np.array(out)

print("fig-power: values plotted")
print(f"  this study: n_positives={n_pos_study}, n_negatives={n_neg_study}, alpha={alpha_study}, power={power_study}")
mde_study = hanley_mde([n_pos_study], [n_neg_study])[0]
print(f"  this study minimum detectable AUROC, stored in the file: {mda:.4f}")
print(f"  this study minimum detectable AUROC, from the plotted curve: {mde_study:.4f}  <- the marker uses this")
print("    (the stored value is not reproduced by this curve; the marker is placed on the curve")
print("     the axes actually draw, so the point and the line agree)")
print(f"  mortality task (task_runs.json / concat_s1 / data_lineage): test_n={mort_n_test}, test_n_pos={mort_n_pos}, test_n_neg={mort_n_neg}")
mort_mde = hanley_mde([mort_n_pos], [mort_n_neg])[0]
print(f"  mortality task minimum detectable AUROC: {mort_mde:.4f}")

n_pos_grid = np.logspace(np.log10(5), np.log10(3000), 60)
neg_pos_ratio = n_neg_study / n_pos_study   # this evaluation set's own positive:negative ratio
n_neg_grid = n_pos_grid * neg_pos_ratio
mde_grid = hanley_mde(n_pos_grid, n_neg_grid)

fig-power: values plotted
  this study: n_positives=12, n_negatives=241, alpha=0.05, power=0.8
  this study minimum detectable AUROC, stored in the file: 0.7382
  this study minimum detectable AUROC, from the plotted curve: 0.7355  <- the marker uses this
    (the stored value is not reproduced by this curve; the marker is placed on the curve
     the axes actually draw, so the point and the line agree)
  mortality task (task_runs.json / concat_s1 / data_lineage): test_n=6776, test_n_pos=570, test_n_neg=6206
  mortality task minimum detectable AUROC: 0.5360


**Step 2. The figure.** Drawn only from the values printed above, then written to `fig-power.pdf`.

In [ ]:
# =============================================================================
# Fig 7 -- fig-power
# =============================================================================
# The significance level is named alpha_level, not alpha: this set bans the
# matplotlib alpha= keyword outright and verification greps for it, so a
# statistical alpha must not wear the same name as a transparency.
def hanley_mde(n_pos, n_neg, alpha_level=0.05, power=0.80):
    """Smallest AUROC distinguishable from 0.5 at the given power, Hanley and
    McNeil variance. Uses BOTH class counts."""
    from scipy.stats import norm
    z = norm.ppf(1 - alpha_level / 2) + norm.ppf(power)
    out = []
    for m_, n_ in zip(np.atleast_1d(n_pos), np.atleast_1d(n_neg)):
        lo, hi = 0.5, 0.9999
        for _ in range(80):
            a = (lo + hi) / 2
            q1, q2 = a / (2 - a), 2 * a * a / (1 + a)
            v = (a * (1 - a) + (m_ - 1) * (q1 - a * a) + (n_ - 1) * (q2 - a * a)) / (m_ * n_)
            if (a - 0.5) / math.sqrt(max(v, 1e-12)) < z:
                lo = a
            else:
                hi = a
        out.append((lo + hi) / 2)
    return np.array(out)


def make_fig_power():
    # Every quantity is read from the register by key: the study's class counts,
    # alpha and power (power.*, whose artifact is section6_auroc_power.json), and
    # the simulated 80 percent power floors (sizectl108.power_curves.*).
    n_pos_study, n_neg_study = fact("power.n_positives"), fact("power.n_negatives")
    alpha_study, power_study = fact("power.alpha"), fact("power.power")
    assert fact("sizectl108.power_curves.n_pos_slots") == n_pos_study
    assert fact("sizectl108.power_curves.n_neg_slots") == n_neg_study
    floor_mw = fact("sizectl108.power_curves.label_mix.crossings_at_power_0_80.mw_two_sided")
    floor_rule = fact("sizectl108.power_curves.label_mix.crossings_at_power_0_80.c98_rule")
    assert floor_mw == fact("sizectl108.power_curves.primary_crossing_power_0_80")
    n_pos = np.logspace(np.log10(5), np.log10(3000), 120)
    n_neg = n_pos * (n_neg_study / n_pos_study)
    mde = hanley_mde(n_pos, n_neg, alpha_level=alpha_study, power=power_study)
    mde_study = hanley_mde([n_pos_study], [n_neg_study], alpha_level=alpha_study,
                           power=power_study)[0]
    assert abs(mde_study - fact("sizectl108.power_curves.mde_analytic")) < 5e-5
    pct = f"{power_study * 100:.0f} percent"
    chance = 0.5

    fig, ax = plt.subplots(figsize=(FIGW, 3.30))
    canvas(fig)
    fig.subplots_adjust(left=0.1172, right=0.9890, top=0.8150, bottom=0.1420)
    Y_LO, Y_HI = 0.47, 0.92
    curve = np.clip(mde, chance, 1.0)
    # Below the curve, between chance and the analytic minimum detectable
    # AUROC, is what this design could not resolve at the stated power: FOG.
    # Above the curve is bare paper. tools/measure_ink.py holds this canvas to
    # a 0.53 coverage floor set when both regions were filled; this canvas
    # measures under it, and that is recorded in manifest.json rather than
    # met by tinting the upper region, which would be a band drawn for the tool.
    ax.fill_between(n_pos, chance, curve, facecolor=FOG, edgecolor="none",
                    linewidth=0, zorder=1)
    ax.plot(n_pos, curve, color=GRAPHITE, lw=2.2, zorder=4)
    ref_tone = SLATE    # reference and guide lines only; never type
    ax.axhline(chance, color=ref_tone, lw=LW_REF, linestyle=REF_DASH, zorder=3)
    ax.text(2800, chance - 0.006, f"chance, AUROC {chance:.2f}", ha="right",
            va="top", fontsize=T_NOTE, color=MUTE, style="italic", zorder=5)
    ax.axvline(n_pos_study, color=ref_tone, lw=0.6, zorder=2)
    ax.text(n_pos_study * 1.04, Y_HI - 0.010, f"{n_pos_study} positives",
            ha="left", va="top", fontsize=T_NOTE, color=MUTE, zorder=5)
    ax.set_xscale("log")
    ax.set_xlim(5, 3000)
    ax.set_ylim(Y_LO, Y_HI)
    ax.set_yticks([0.5, 0.6, 0.7, 0.8, 0.9])
    ax.xaxis.set_major_formatter(
        matplotlib.ticker.FuncFormatter(lambda v, _: f"{v:,.0f}"))
    ax.xaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())
    ax.tick_params(axis="both", labelsize=T_TICK)
    ax.set_xlabel("Positive examples in the evaluation set (log scale)",
                  fontsize=T_AXIS)
    ax.set_ylabel(f"Minimum detectable AUROC\n({pct} power)", fontsize=T_AXIS,
                  linespacing=1.3)
    ax.text(2800, 0.60, f"detectable at {pct} power", ha="right",
            va="bottom", fontsize=T_SERIES, color=MUTE, zorder=5)
    ax.text(14, 0.508, f"not detectable at {pct} power", ha="left",
            va="bottom", fontsize=T_SERIES, color=MUTE, zorder=5)

    # The three points at this study's positive count, on the guide.
    ax.plot([n_pos_study], [mde_study], "o", ms=6, markerfacecolor=PAPER,
            markeredgecolor=INK, markeredgewidth=1.1, zorder=7)
    ax.plot([n_pos_study], [floor_mw], "o", ms=5.2, color=INK,
            markeredgewidth=0, zorder=8)
    ax.plot([n_pos_study], [floor_rule], "D", ms=5.6, color=INK,
            markeredgewidth=0, zorder=8)
    # Labels to the right, each joined to its point by a thin leader.
    x_lab = 30.0
    for y_pt, y_lab, text in (
            (floor_rule, floor_rule + 0.025,
             f"registered decision rule, power simulation: {floor_rule:.2f}"),
            (floor_mw, floor_mw + 0.035,
             f"Mann-Whitney test, power simulation: {floor_mw:.2f}"),
            (mde_study, 0.552, f"analytic curve: {mde_study:.2f}")):
        ax.plot([n_pos_study * 1.10, x_lab * 0.95], [y_pt, y_lab],
                color=GRAPHITE, lw=0.6, zorder=6, solid_capstyle="butt")
        ax.text(x_lab, y_lab, text, ha="left", va="center", fontsize=T_SERIES,
                color=GRAPHITE, zorder=7)
    frame(ax)
    signature(fig, INK,
              f"At {n_pos_study} positives, {pct} power needs AUROC "
              f"{floor_mw:.2f} under the Mann-Whitney test\nand {floor_rule:.2f} "
              f"under the registered decision rule.")
    fig.savefig(HERE / "fig-power.pdf"); plt.close(fig)
    print("wrote fig-power.pdf")


if __name__ == "__main__":
    make_fig_prevalence_response()
    make_fig_scaling()
    make_fig_forest()
    make_fig_power()


## Figure 8: `fig-forest`


In [ ]:
iv = json.load(open(os.path.join(
    RESULTS_ACCORD, "stats", "intervals.json")))["E2_mcc_intervals"]
dec = json.load(open(os.path.join(
    RESULTS_ACCORD, "decontam", "corrected_gold_metrics.json")))["aggregate"]
df34 = json.load(open(os.path.join(
    RESULTS_ACCORD, "control", "final_report.json")))[
        "ensemble_mean_of_seed_scores"]["gold_defect_free_rule34"]
tr = json.load(open(os.path.join(RESULTS, "task_runs.json")))
sc = json.load(open(os.path.join(
    RESULTS_ACCORD, "scaling", "scaling_results.json")))

print("fig-forest: every headline estimate in the thesis, read against zero")
for k, v in iv.items():
    print(f"  MCC {k:32s} {v['mcc']:+.4f}  [{v['ci95'][0]:+.4f}, {v['ci95'][1]:+.4f}]")
print(f"  AUROC decontaminated ensemble        {dec['ensemble_auroc']-0.5:+.4f}"
      f"  [{dec['ensemble_ci95'][0]-0.5:+.4f}, {dec['ensemble_ci95'][1]-0.5:+.4f}]")
print(f"  AUROC defect-free stratum            {df34['auroc']-0.5:+.4f}"
      f"  [{df34['bootstrap']['ci95'][0]-0.5:+.4f}, {df34['bootstrap']['ci95'][1]-0.5:+.4f}]")
print("  pretraining gaps are recomputed in the figure cell from task_runs.json"
      " and scaling_results.json")


**Step 2. The figure.** Drawn only from the values printed above, then written to `fig-forest.pdf`.

In [ ]:
# =============================================================================
# Fig 8 -- fig-forest. The thesis's headline estimates on the clinical pairs,
# one panel per unit.
#
# ROUND 7 REBUILD (fixes_r7 N9). The previous canvas headlined "5 of the 6
# headline estimates cross zero", and two of the six were the C44 pretraining
# arms, which move to the supplement because their registration forbids using
# them on this evaluation, while a third was the Matthews correlation of the
# contaminated released run. It now draws what the thesis currently calls its
# headline estimates, and nothing else:
#   AUROC   the decontaminated detector ensemble on the 253 non-ambiguous pairs,
#           with its bootstrap interval, and the same ensemble on the Rules 3
#           and 4 stratum with ambiguous pairs dropped (108 pairs), with the
#           C114 bootstrap interval. The stratum is not labelled defect-free:
#           its exemplar correctness is assumed, not verified.
#   MCC     the deployed weak labeller measured by the C120 exact join on 299
#           pairs, drawn against the ceiling its own marginal sets. A count on
#           a fixed table has no resampling interval here; it carries the exact
#           one-sided hypergeometric p instead, and the canvas says so.
# The two units get two rulers, as before, because an AUROC and an MCC are not
# comparable by horizontal distance. What the panels share is the claim: none
# of the three is distinguishable from chance.
# =============================================================================
def make_fig_forest():
    n_253 = fact("decontam.n")
    n_108 = fact("c114.task_c.n")
    n_299 = fact("c120.g299.n")
    assert n_253 == fact("c120.g253.n")
    au = [
        (f"Ensemble, {n_253} non-ambiguous pairs",
         fact("decontam.aggregate.ensemble_auroc"),
         fact("decontam.aggregate.ensemble_ci95_lo"),
         fact("decontam.aggregate.ensemble_ci95_hi")),
        (f"Ensemble, Rules 3 and 4, {n_108} pairs",
         fact("c114.task_c.stratum_auroc"),
         fact("c114.task_c.ci95_lo"), fact("c114.task_c.ci95_hi")),
    ]
    assert all(lo <= 0.5 <= hi for _, _, lo, hi in au), \
        "an AUROC interval now excludes chance; the headline would be false"
    mcc_obs = fact("c120.g299.mcc")
    mcc_ceil = fact("c120.g299.ceiling_mcc")
    mcc_p = fact("c120.g299.p_hypergeom_one_sided")
    assert mcc_p > 0.05
    floor = fact("sizectl108.power_curves.primary_crossing_power_0_80")
    power = fact("power.power")
    n_pos_floor = fact("sizectl108.power_curves.n_pos_slots")
    assert n_pos_floor == fact("decontam.n_positives")

    H = 2.80
    fig = plt.figure(figsize=(FIGW, H))
    canvas(fig)
    X_LEFT, X_RIGHT, X_VALUE = 0.3250, 0.7700, 0.9950
    A0, AH = 1.24, 0.84
    M0, MH = 0.36, 0.40
    ax_a = fig.add_axes([X_LEFT, A0 / H, X_RIGHT - X_LEFT, AH / H])
    ax_m = fig.add_axes([X_LEFT, M0 / H, X_RIGHT - X_LEFT, MH / H])

    # AUROC panel: chance at 0.5 is the zero rule; the part of the axis above
    # the 80 percent power floor at this study's class counts is shaded.
    YA_LO, YA_HI = -0.8, 2.5
    ax_a.set_xlim(0.28, 1.0)
    ax_a.set_ylim(YA_LO, YA_HI)
    ax_a.set_xticks([0.3, 0.5, 0.7, 0.9])
    frame(ax_a, grid="x")
    ax_a.axvspan(floor, 1.0, facecolor=FOG, edgecolor="none", lw=0, zorder=0)
    ax_a.text((floor + 1.0) / 2, YA_HI - 0.18,
              f"detectable at\n{power * 100:.0f} percent power\nwith {n_pos_floor} positives",
              ha="center",
              va="top", fontsize=T_NOTE, color=MUTE, linespacing=1.15,
              zorder=3)
    ax_a.axvline(0.5, color=GRAPHITE, lw=1.2, zorder=2)
    ax_a.text(0.492, YA_LO + 0.08, "chance", ha="right", va="bottom",
              fontsize=T_NOTE, color=MUTE, style="italic")
    for i, (lab, e, lo, hi) in enumerate(au):
        y = len(au) - 1 - i
        ax_a.errorbar([e], [y], xerr=[[e - lo], [hi - e]], fmt="o",
                      color=ENSEMBLE_DEEP, markerfacecolor=ENSEMBLE_CORE,
                      markeredgecolor=ENSEMBLE_DEEP, zorder=5,
                      **dict(ERRBAR, elinewidth=1.8, capthick=1.8), **DOT)
        fig.text(X_VALUE, (A0 + AH * (y - YA_LO) / (YA_HI - YA_LO)) / H,
                 f"{e:.2f}  [{lo:.2f}, {hi:.2f}]", ha="right", va="center",
                 fontsize=T_SERIES, color=ENSEMBLE_DEEP)
    ax_a.set_yticks([len(au) - 1 - i for i in range(len(au))])
    ax_a.set_yticklabels([r[0] for r in au], fontsize=T_SERIES)
    ax_a.tick_params(axis="y", length=0)
    ax_a.tick_params(axis="x", labelsize=T_TICK)
    fig.text(0.012, (A0 + AH + 0.08) / H,
             "AUROC, decontaminated detector, 95% bootstrap interval",
             ha="left", va="bottom", fontsize=T_SERIES, color=MUTE,
             style="italic")

    # MCC panel: zero is no association; the ceiling is a bound, drawn as a tick.
    ax_m.set_xlim(-0.02, 0.10)
    ax_m.set_ylim(-0.6, 0.6)
    ax_m.set_xticks([0.0, 0.03, 0.06, 0.09])
    frame(ax_m, grid="x")
    ax_m.axvline(0.0, color=GRAPHITE, lw=1.2, zorder=2)
    ax_m.text(0.0015, -0.56, "no association", ha="left", va="bottom",
              fontsize=T_NOTE, color=MUTE, style="italic")
    ax_m.plot([mcc_ceil, mcc_ceil], [-0.38, 0.38], color=WEAK_LABELLER, lw=2.0,
              solid_capstyle="butt", zorder=4)
    ax_m.plot([0.0, mcc_ceil], [0, 0], color=HAIR, lw=3.0, zorder=3,
              solid_capstyle="butt")
    ax_m.plot([mcc_obs], [0], "o", color=WEAK_LABELLER,
              markeredgecolor=WEAK_LABELLER, zorder=5, **DOT)
    ax_m.text(mcc_ceil + 0.002, 0.30, "ceiling", ha="left", va="top",
              fontsize=T_NOTE, color=MUTE, style="italic")
    ax_m.set_yticks([0])
    ax_m.set_yticklabels([f"Deployed labeller, {n_299} pairs"], fontsize=T_SERIES)
    ax_m.tick_params(axis="y", length=0)
    ax_m.tick_params(axis="x", labelsize=T_TICK)
    fig.text(X_VALUE, (M0 + MH / 2) / H,
             f"{mcc_obs:.2f}  ceiling {mcc_ceil:.2f}\nexact p {mcc_p:.2f}",
             ha="right", va="center", fontsize=T_SERIES, color=WEAK_LABELLER,
             linespacing=1.25)
    fig.text(0.012, (M0 + MH + 0.08) / H,
             "MCC against the pharmacist, exact count, no resampling interval",
             ha="left", va="bottom", fontsize=T_SERIES, color=MUTE,
             style="italic")
    fig.add_artist(plt.Line2D([0.012, X_VALUE], [1.02 / H] * 2,
                              color=PLATE_EDGE, lw=0.6))

    n_pos_253 = fact("decontam.n_positives")
    assert n_pos_253 == fact("c120.g299.n_positives")
    signature(fig, GRAPHITE,
              f"At {n_pos_253} positives, or {fact('c114.task_c.n_pos')} for Rules 3 "
              "and 4, no headline estimate is distinguishable from chance.",
              deck="Two units, two rulers: nothing is comparable across the "
                   "panel boundary.")
    fig.savefig(HERE / "fig-forest.pdf"); plt.close(fig)
    print("wrote fig-forest.pdf")


## Figure 9: `fig-scaling`


In [21]:
p_scaling = os.path.join(RESULTS_ACCORD, "scaling", "scaling_results.json")
d_scaling = json.load(open(p_scaling))
by, params_by_tag = {}, {}
for x in d_scaling:
    by.setdefault(x["tag"], []).append(x["accuracy"])
    params_by_tag[x["tag"]] = x["non_emb_params"]
tags = ["scratch_10M", "scratch_30M", "scratch_100M"]
xs = [params_by_tag[t] for t in tags]
mu = [statistics.mean(by[t]) for t in tags]
sd = [statistics.stdev(by[t]) for t in tags]
pm, ps, pp = statistics.mean(by["pretrained_bert"]), statistics.stdev(by["pretrained_bert"]), params_by_tag["pretrained_bert"]

p_task_runs = os.path.join(RESULTS, "task_runs.json")
d_task_runs = json.load(open(p_task_runs))
pb = [d_task_runs[f"concat_s{s}"]["auroc"] for s in (1, 2, 3)]
ou = [d_task_runs[f"ours_current_s{s}"]["auroc"] for s in (0, 1, 2)]
pbm, pbs = statistics.mean(pb), statistics.stdev(pb)
oum, ous = statistics.mean(ou), statistics.stdev(ou)
gap = pbm - oum
clinical_params = d_task_runs["ours_current_s0"]["non_emb_params"]

print("fig-scaling: values plotted")
print(f"  AG News: pretrained_bert mean +/- std = {pm:.4f} +/- {ps:.4f}  ({pp / 1e6:.0f}M params)")
for t, x, m, s in zip(tags, xs, mu, sd):
    print(f"  AG News: {t:14s} mean +/- std = {m:.4f} +/- {s:.4f}  ({x / 1e6:.1f}M params)  gap to pretrained = {pm - m:+.4f}")
print(f"  clinical task: PubMedBERT (concat_s1..3) AUROC mean +/- std = {pbm:.4f} +/- {pbs:.4f}")
print(f"  clinical task: from-scratch (ours_current_s0..2) AUROC mean +/- std = {oum:.4f} +/- {ous:.4f}  ({clinical_params / 1e6:.1f}M params)")
print(f"  clinical task gap: {gap:+.4f}")

fig-scaling: values plotted
  AG News: pretrained_bert mean +/- std = 0.9396 +/- 0.0010  (86M params)
  AG News: scratch_10M    mean +/- std = 0.8790 +/- 0.0040  (10.6M params)  gap to pretrained = +0.0606
  AG News: scratch_30M    mean +/- std = 0.8833 +/- 0.0009  (31.9M params)  gap to pretrained = +0.0563
  AG News: scratch_100M   mean +/- std = 0.8878 +/- 0.0016  (99.2M params)  gap to pretrained = +0.0518
  clinical task: PubMedBERT (concat_s1..3) AUROC mean +/- std = 0.7693 +/- 0.0102
  clinical task: from-scratch (ours_current_s0..2) AUROC mean +/- std = 0.7662 +/- 0.0073  (10.6M params)
  clinical task gap: +0.0031


**Step 2. The figure.** Drawn only from the values printed above, then written to `fig-scaling.pdf`.

In [ ]:
# =============================================================================
# Fig 9 -- fig-scaling. RETIRED, and still built here.
#
# The figure is three points and a line, every number of which is already in a
# table, and its terminal claim -- the pretrained arm beats the 100M scratch
# arm -- belongs to the C44 control, which round 7 moved to the supplement, so
# fig:forest no longer carries it either. Nothing in the thesis binds it any more and the
# reason of record is in tools/check_figure_invariants.py.
#
# It goes on being built, rather than being switched off the way fig_frontier.py
# and make_figures_extra.py were, because this module still builds three live
# figures and because that is what this directory already does with a retired
# figure whose generator shares a file: make_figures.py has kept building
# fig-funnel, fig-prevalence and fig-tracking since each of them was retired.
# The retirement record is what keeps a retired PDF out of the thesis; a
# generator that stops running only makes the PDF stale.
#
# Panel (b) was deleted earlier: its content is Figure 8's last row.
# =============================================================================
def make_fig_scaling():
    sc = json.loads((ROOT / "results/accord/scaling/scaling_results.json").read_text())
    by, pr_ = {}, {}
    for x in sc:
        by.setdefault(x["tag"], []).append(x["accuracy"]); pr_[x["tag"]] = x["non_emb_params"]
    tags = ["scratch_10M", "scratch_30M", "scratch_100M"]
    xs = [pr_[t] for t in tags]; mu = [st.mean(by[t]) for t in tags]
    sd = [st.stdev(by[t]) for t in tags]
    pm, ps, pp = st.mean(by["pretrained_bert"]), st.stdev(by["pretrained_bert"]), pr_["pretrained_bert"]

    fig, ax = plt.subplots(figsize=(FIGW, 2.84))
    canvas(fig)
    # Absolute margins held; the whole of the canvas's growth goes to the plot.
    fig.subplots_adjust(left=0.1209, right=0.9890, top=0.7904, bottom=0.1648)
    # The pretrained arm is a measured arm with three seeds and a spread of its
    # own, not a chance line, so its band, its centre line and its label are all
    # one hue and that hue is the reference-standard green.
    ax.axhspan(pm - ps, pm + ps, facecolor=EMERALD_WASH, edgecolor="none",
               linewidth=0, zorder=1)
    ax.axhline(pm, color=EMERALD_CORE, lw=LW_DATA, zorder=2)
    # The gap between the two arms IS this figure's subject, and it was white
    # space. It is the region between two measured series, so it is drawn as one.
    ax.fill_between(xs, mu, [pm] * len(xs), facecolor=MIST, edgecolor="none",
                    linewidth=0, zorder=1)
    ax.errorbar(xs, mu, yerr=sd, fmt="none", ecolor=COBALT_CORE, zorder=3, **ERRBAR)
    # SEED DOTS ARE DRAWN ON THE FROM-SCRATCH ARM ONLY, and the pretrained arm is
    # the reason. Size is the variable on the scratch arm, so each of its points has
    # an x and its three seeds stack on it. The pretrained arm is not a point on
    # that sweep: it is a reference level, which is why it is drawn as a band across
    # the WHOLE width rather than a mark at 8.6e7. Its three seeds are already on
    # the canvas twice over -- they are what the band's half-height and the label's
    # "plus or minus 0.0010" are computed from. Drawn a third time as dots they
    # landed on the band's own centre line: two vanished under the 1.8pt stroke and
    # the third cleared it by 2pt, leaving one isolated slate fleck grazing the
    # label. Z-order cannot fix that (the third seed lies outside mean +/- sd, so it
    # clears the band as well as the line) and jitter would put a false parameter
    # count under a log axis. The band is the honest drawing of these three numbers.
    for x, t in zip(xs, tags):
        ax.plot([x] * len(by[t]), by[t], "o", color=SLATE, ms=MS_SEED,
                markeredgewidth=0, zorder=4)
    ax.plot(xs, mu, "-o", color=COBALT_CORE, ms=MS_DOT, lw=LW_DATA, zorder=5)
    for x, m_, s_ in zip(xs, mu, sd):
        ax.annotate(f"{pm - m_:+.4f}", (x, m_ - s_), textcoords="offset points",
                    xytext=(0, -11), ha="center", va="top", fontsize=T_NOTE,
                    color=MUTE)
    ax.set_xscale("log")
    ax.set_xlabel("Non-embedding parameters (log scale)")
    ax.set_ylabel("Accuracy, AG News")
    ax.set_ylim(0.851, 0.950)
    ax.set_yticks([0.86, 0.88, 0.90, 0.92, 0.94])
    # The xlim runs 8% further right than the pretrained point, which is the room
    # the label needs to sit inside the box at the line's own right end.
    ax.set_xlim(7.0e6, 2.1e8)
    case(ax.text(2.0e8, pm - 0.0034,
                 f"pretrained BERT {pm:.4f} plus or minus {ps:.4f}",
                 fontsize=T_SERIES, color=EMERALD_DEEP, va="top", ha="right",
                 zorder=6))
    # The subject of the figure, named inside itself. MUTE on MIST is 4.9:1.
    ax.text(5.0e7, 0.9150, "the gap", fontsize=T_NOTE, color=MUTE,
            ha="center", va="center", zorder=6)
    frame(ax, grid=True)
    ax.plot(xs, mu, "-o", color=COBALT_CORE, ms=MS_DOT, lw=LW_DATA, zorder=5)
    case(ax.text(xs[1], mu[1] + 0.0060, "from scratch", fontsize=T_SERIES,
                 color=COBALT_DEEP, va="bottom", ha="center", zorder=6))
    signature(fig, EMERALD_DEEP,
              "The from-scratch arm rises with size\n"
              "and never reaches the pretrained band.")
    fig.savefig(HERE / "fig-scaling.pdf"); plt.close(fig)
    print("wrote fig-scaling.pdf")


## Figure 10: `fig-threshold-surface`


In [23]:
p_thresh = os.path.join(RESULTS_ACCORD, "selection", "threshold_robustness_results.json")
d_thresh = json.load(open(p_thresh))
df = pd.DataFrame(d_thresh["cells"])
piv = df.pivot_table(index="achieved_rate", columns="beta", values="demographic_parity_gap", aggfunc="mean")
mde = df.pivot_table(index="achieved_rate", columns="beta", values="mde_demographic_parity_gap", aggfunc="mean")
R, B, Z, M = piv.index.values, piv.columns.values, piv.values, mde.values
e9_rate = d_thresh["original_e9_population_pred_pos_rate"]

# Named setting, not read from this file: the audit's own demographic-parity tolerance.
TOLERANCE = 0.05

print("fig-threshold-surface: values plotted")
print(f"  grid: {Z.shape[0]} operating points x {Z.shape[1]} selection strengths = {Z.size} cells")
print(f"  demographic-parity gap range: [{np.nanmin(Z):.4f}, {np.nanmax(Z):.4f}]")
print(f"  minimum-detectable-gap range: [{np.nanmin(M):.4f}, {np.nanmax(M):.4f}]")
n_under = int((M > TOLERANCE).sum())
print(f"  cells underpowered at TOLERANCE={TOLERANCE}: {n_under} of {M.size}")
print(f"  original E9 operating point (predicted-positive rate): {e9_rate:.4f}")

# Viridis: perceptually uniform, monotone in lightness end to end, and safe
# under every common dichromacy, which is why it is the standard for a field
# like this one. It replaces a five-stop ramp in a single blue. The ramp was
# defensible and it was also the reason a surface of 270 measured cells read as
# one wash: a single hue can only spend lightness, so the interesting middle of
# this field, between about 0.04 and 0.12, occupied one narrow band of pale
# blue. Viridis spends hue as well, so the same middle is a visible traverse
# from blue through teal to green, and the greyscale reading survives because
# viridis is still monotone in lightness: dark is low, light is high, in print
# and on paper both.
cmap = plt.get_cmap("viridis")

fig-threshold-surface: values plotted
  grid: 18 operating points x 15 selection strengths = 270 cells
  demographic-parity gap range: [0.0021, 0.2001]
  minimum-detectable-gap range: [0.0305, 0.0600]
  cells underpowered at TOLERANCE=0.05: 195 of 270
  original E9 operating point (predicted-positive rate): 0.1994


**Step 2. The figure.** Drawn only from the values printed above, then written to `fig-threshold-surface.pdf`.

In [ ]:
ROOT = pathlib.Path("/Users/rickygole/provenance-encoder")
d_thresh = json.loads((ROOT / "results/accord/selection/"
                       "threshold_robustness_results.json").read_text())
df = pd.DataFrame(d_thresh["cells"])
piv = df.pivot_table(index="achieved_rate", columns="beta",
                     values="demographic_parity_gap", aggfunc="mean")
mde = df.pivot_table(index="achieved_rate", columns="beta",
                     values="mde_demographic_parity_gap", aggfunc="mean")
R, B, Z, M = piv.index.values, piv.columns.values, piv.values, mde.values
e9_rate = d_thresh["original_e9_population_pred_pos_rate"]

# Named setting: the audit's own demographic-parity tolerance.
TOLERANCE = 0.05

plt.rcParams["hatch.color"] = SLATE
plt.rcParams["hatch.linewidth"] = 0.7

# 2.85in, not the 3.06 a proportional growth would give: at 0.98 of the measure
# 3.06 puts the block at 4.50in and this figure into class B, and its whole
# design brief was a single-page-share float. The width takes the growth
# instead, and the heatmap field goes from 3.28in wide to 3.77in, which is
# where the illegibility actually was.
fig = plt.figure(figsize=(FIGW, 2.85))
canvas(fig)
ax = fig.add_axes([0.08973, 0.20828, 0.64473, 0.60002])
# The colourbar sits 0.017 further left than an absolute-margin conversion puts
# it: its rotated label plus its widest tick ran 6.5pt past the right edge and
# the tight box grew to catch them, which costs the drawn-at-printed-size
# invariant the whole geometry rests on.
cax = fig.add_axes([0.89000, 0.20828, 0.02564, 0.60002])
im = ax.pcolormesh(B, R, Z, cmap=plt.get_cmap("viridis"), shading="nearest",
                   vmin=0, vmax=0.20)

under = (M > TOLERANCE).astype(float)
# A different channel, not a weaker version of the same one.
ax.contourf(B, R, under, levels=[0.5, 1.5], colors=[MIST], zorder=2)
ax.contourf(B, R, under, levels=[0.5, 1.5], colors=["none"], hatches=["////"],
            zorder=2.1)
ax.contour(B, R, under, levels=[0.5], colors=[GRAPHITE], linewidths=1.2,
           zorder=3)
# THE SMALL CAP ON THE BOTTOM EDGE NEAR beta = 0.5 IS DATA, NOT A LABELLING
# ARTIFACT. At the lowest predicted-positive rate the parity gap runs
# 0.0132, 0.0543, 0.0295 across beta = 0, 0.5, 1.0, so it crosses the 0.05
# tolerance up and back down inside one column. contour returns it as its own
# three-point segment, [(0.448, 0.090), (0.500, 0.106), (0.586, 0.090)], and it
# is drawn for the same reason the long branches are. Verified by rendering
# allsegs with no labelling of any kind in the figure. Do not delete it.
cs_tol = ax.contour(B, R, Z, levels=[TOLERANCE], colors=[GRAPHITE],
                    linewidths=1.2, zorder=4)

ax.set_xlabel(r"Selection strength through documentation volume ($\beta$)")
ax.set_ylabel("Predicted-positive rate")
ax.set_xticks([0, 2, 4, 6, 8])
ax.set_yticks([0.1, 0.2, 0.3, 0.4, 0.5, 0.6])
ax.set_xlim(B.min(), B.max())
ax.set_ylim(R.min(), R.max())
frame(ax)

# The E9 operating point: the one scarlet element, and it is the claim.
ax.plot(B, [e9_rate] * len(B), color=SCARLET_CORE, lw=LW_DATA, zorder=6)
case(ax.text(B[-1], e9_rate + 0.012, "the audit's operating point",
             ha="right", va="bottom", fontsize=T_SERIES, color=SCARLET_DEEP,
             zorder=7))


def exit_y(cset):
    """The y at which a contour leaves the plot on the right, so its label can
    sit in the margin beside it instead of riding the line."""
    best = None
    for seg in cset.allsegs[0]:
        if len(seg) == 0:
            continue
        i = int(np.argmax(seg[:, 0]))
        if best is None or seg[i, 0] > best[0]:
            best = (seg[i, 0], seg[i, 1])
    return None if best is None else best[1]


def exit_x_top(cset, y_top, tol=1e-9):
    """The x values at which a contour leaves the plot through the TOP edge.

    The tolerance contour never reaches the right-hand spine: both of its
    branches run off the top, at beta = 0.73 and beta = 2.59. Labelled at the
    right margin it pointed at a spine the line does not touch, so the label
    goes above the top edge at the branch's own exit x instead.
    """
    out = []
    for seg in cset.allsegs[0]:
        for x, y in seg:
            if abs(y - y_top) <= tol + 1e-12:
                out.append(float(x))
    return sorted(out)


AX_X0, AX_Y0, AX_W, AX_H = 0.08973, 0.20828, 0.64473, 0.60002
y_pow = exit_y(ax.contour(B, R, under, levels=[0.5], colors=["none"],
                          linewidths=0, zorder=0))
margin_x = 0.74170
if y_pow is not None:
    yf = AX_Y0 + AX_H * (y_pow - R.min()) / (R.max() - R.min())
    fig.text(margin_x, yf, "power limit", ha="left", va="center",
             fontsize=T_SERIES, color=GRAPHITE)
    fig.patches.append(plt.Rectangle(
        (0.73446, yf - 0.0022), 0.005467, 0.0044, transform=fig.transFigure,
        facecolor=GRAPHITE, edgecolor="none", zorder=8))

tol_x = exit_x_top(cs_tol, R.max())
if tol_x:
    # The rightmost of the two top exits: the left one at beta = 0.73 would
    # centre a 51pt label off the left spine.
    xt = tol_x[-1]
    xf = AX_X0 + AX_W * (xt - B.min()) / (B.max() - B.min())
    fig.patches.append(plt.Rectangle(
        (xf - 0.0010, AX_Y0 + AX_H), 0.0020, 3.0 / (72 * 2.85),
        transform=fig.transFigure, facecolor=GRAPHITE, edgecolor="none",
        zorder=8))
    fig.text(xf, AX_Y0 + AX_H + 5.0 / (72 * 2.85), f"{TOLERANCE} tolerance",
             ha="center", va="bottom", fontsize=T_SERIES, color=GRAPHITE)

# The inability statement is carried on canvas by the headline and by the
# hatching itself; the sentence naming the hatched region is the caption's job
# and the caption says it verbatim (msu_ch5_discussion.tex). It was set twice.

cb = fig.colorbar(im, cax=cax)
cb.set_label("Demographic-parity gap", fontsize=T_AXIS)
cb.set_ticks([0, 0.05, 0.10, 0.15, 0.20])
cb.ax.tick_params(labelsize=T_TICK, width=LW_SPINE, length=3.2)
cb.outline.set_visible(True)
cb.outline.set_linewidth(LW_SPINE)
cb.outline.set_edgecolor(INK)

signature(fig, SCARLET_DEEP,
          "The audit passed at an operating point where nothing was detectable.")
out = HERE / "fig-threshold-surface.pdf"
fig.savefig(out)
n_under = int((M > TOLERANCE).sum())
print(f"wrote {out.name}  ({Z.shape[0]}x{Z.shape[1]} cells, {n_under} underpowered)")


## Figure 11: `fig-disc-actionability`


In [25]:
print("fig-disc-actionability: no result file is read by this cell.")
print("This figure is a schematic of a MIMIC-III field-availability fact (accord/CORRECTIONS.md, C6):")
print("  nursing notes carry CHARTTIME (a timestamp); discharge summaries carry only CHARTDATE (a date).")
print("No numeric estimate is plotted, so there is no CI to attach and nothing to source from a JSON file.")

# Widened from 10.5 to 11.6 units so the longest event label no longer runs off
# the right-hand edge, and every leader now starts outside the shaded band
# instead of crossing it. The trigger point carries one marker, not an X drawn
# on top of a dot.

fig-disc-actionability: no result file is read by this cell.
This figure is a schematic of a MIMIC-III field-availability fact (accord/CORRECTIONS.md, C6):
  nursing notes carry CHARTTIME (a timestamp); discharge summaries carry only CHARTDATE (a date).
No numeric estimate is plotted, so there is no CI to attach and nothing to source from a JSON file.


**Step 2. The figure.** Drawn only from the values printed above, then written to `fig-disc-actionability.pdf`.

In [ ]:
W, H = FIGW, 2.13
fig, ax = plt.subplots(figsize=(W, H))
canvas(fig)
plate(fig)
fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
ax.set_xlim(0, W * 72)
ax.set_ylim(0, H * 72)
ax.axis("off")

# The point grid moves with the plate, which is 9 percent larger in both
# directions than it was; the type on it does not move, so the timeline
# gains air rather than size.
x_nurse, x_order, x_disch, x_summary = 52.0, 164.0, 293.0, 367.0
x0, x1 = 1.5, W * 72 - 1.5
g_lo, g_hi = 11.0, 122.0
band_lo, band_hi = 57.0, 74.0          # the phase band
mid = (band_lo + band_hi) / 2

phases = [
    (x0, x_order, EMERALD_WASH, EMERALD_TINT, EMERALD_DEEP,
     "intervention possible"),
    (x_order, x_disch, AMBER_WASH, AMBER_TINT, AMBER_DEEP,
     "window closing"),
    (x_disch, x1, SCARLET_WASH, SCARLET_TINT, SCARLET_DEEP,
     "no intervention possible"),
]
for a, b, wash, tint, deep, name in phases:
    ax.add_patch(mpatches.Rectangle((a, g_lo), b - a, g_hi - g_lo,
                                    facecolor=wash, edgecolor="none",
                                    linewidth=0, zorder=0))
for a, b, wash, tint, deep, name in phases:
    ax.add_patch(mpatches.Rectangle((a, band_lo), b - a, band_hi - band_lo,
                                    facecolor=tint, edgecolor=deep,
                                    linewidth=LW_NODE, zorder=2))
for a, b, wash, tint, deep, name in phases:
    case(ax.text((a + b) / 2, mid, name, ha="center", va="center",
                 fontsize=T_NOTE, color=INK, zorder=6))

# Event ticks pierce the band; names above it on two staggered levels, so
# no two names share a line and none needs a leader.
# The tick meets the band at both edges rather than running through it: the
# inside of the band belongs to the phase name, and a tick crossing that
# name is the one collision this layout can produce.
events = [
    (x_nurse, "last nursing note", GRAPHITE, 7.0, "center"),
    (x_order, "discharge order", GRAPHITE, 22.0, "center"),
    (x_disch, "the patient leaves", GRAPHITE, 7.0, "center"),
    (x_summary, "discharge summary finalised", SCARLET_DEEP, 22.0, "right"),
]
for x, label, tone, dy, ha in events:
    for lo, hi in ((band_lo - 6.0, band_lo), (band_hi, band_hi + 6.0)):
        ax.plot([x, x], [lo, hi], color=tone, lw=LW_CONNECTOR,
                solid_capstyle="butt", zorder=5)
    ax.text(x1 - 4.0 if ha == "right" else x, band_hi + dy, label, ha=ha,
            va="bottom", fontsize=T_SERIES, color=tone, zorder=6)

y_mark = band_lo - 10.0
y_note = band_lo - 22.0
# The point a detector could have fired: a proposal that was measured, so a
# CORE mark in the entity blue rather than an accent.
ax.plot([x_order], [y_mark], "o", color=COBALT_CORE, markersize=MS_MARK,
        markeredgecolor=COBALT_DEEP, markeredgewidth=1.0, zorder=7)
ax.text(x_order, y_note, "proposed detection point", ha="center", va="top",
        fontsize=T_SERIES, color=COBALT_DEEP, zorder=7)

# The system's trigger point: the one saturated scarlet mark on the canvas.
ax.plot([x_summary], [y_mark], "D", color=SCARLET_DEEP, markersize=9.4,
        markeredgecolor=SCARLET_CORE, markeredgewidth=LW_CHIP, zorder=7)
ax.text(x_summary, y_mark, "!", ha="center", va="center", fontsize=T_NOTE,
        color=PAPER, fontweight="bold", zorder=8)
ax.text(x1 - 4.0, y_note, "the system triggers here", ha="right", va="top",
        fontsize=T_SERIES, color=SCARLET_DEEP, zorder=7)

signature(fig, SCARLET_DEEP,
          "The document the system triggers on is finalised at or "
          "after the patient leaves.")
fname = "fig-disc-actionability.pdf"
fig.savefig(os.path.join(FIGDIR, fname))
plt.close(fig)
register(fname, "fig:disc-actionability", ["accord/CORRECTIONS.md (C6)"],
         "Structural/schema diagram, not derived from a numeric result file; documents a "
         "MIMIC-III field-availability fact. No CI applies (no estimate is plotted).")


## Figure 12: `fig-met-exchange`


In [27]:
p_baselines = os.path.join(RESULTS_ACCORD, "diag", "step3_baselines.json")
d_baselines = json.load(open(p_baselines))
systems = {
    "ACCORD": d_baselines["accord_binary_at_0.5_253core"],
    "GPT-4o zero-shot": d_baselines["gpt4o_zeroshot"],
    "GPT-4o few-shot": d_baselines["gpt4o_fewshot"],
    "Llama 3 8B": d_baselines["llama_baseline"],
}
ref = systems["ACCORD"]
TP0, TN0, FP0, FN0 = ref["tp"], ref["tn"], ref["fp"], ref["fn"]
P = TP0 + FN0
N = TN0 + FP0

def mcc_from_counts(tp, tn, fp, fn):
    num = tp * tn - fp * fn
    den = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    return 0.0 if den == 0 else num / den

mcc0 = mcc_from_counts(TP0, TN0, FP0, FN0)

print("fig-met-exchange: values plotted")
print(f"  reference system (ACCORD): TP={TP0}, TN={TN0}, FP={FP0}, FN={FN0}  (P={P}, N={N}, n={P + N})")
print(f"  reference MCC: {mcc0:.4f}")
for name, s in systems.items():
    delta = TP0 - s["tp"]
    dtn = s["tn"] - TN0
    print(f"  {name:18s}: TP={s['tp']:3d} TN={s['tn']:3d} FP={s['fp']:3d} FN={s['fn']:3d}  delta_TP={delta:+d}  delta_TN={dtn:+d}")

fig-met-exchange: values plotted
  reference system (ACCORD): TP=11, TN=18, FP=223, FN=1  (P=12, N=241, n=253)
  reference MCC: -0.0070
  ACCORD            : TP= 11 TN= 18 FP=223 FN=  1  delta_TP=+0  delta_TN=+0
  GPT-4o zero-shot  : TP=  6 TN=176 FP= 65 FN=  6  delta_TP=+5  delta_TN=+158
  GPT-4o few-shot   : TP=  7 TN=107 FP=134 FN=  5  delta_TP=+4  delta_TN=+89
  Llama 3 8B        : TP= 11 TN= 10 FP=231 FN=  1  delta_TP=+0  delta_TN=-8


**Step 2. The figure.** Drawn only from the values printed above, then written to `fig-met-exchange.pdf`.

In [ ]:
baselines, p_baselines = load("diag", "step3_baselines.json")
# The two GPT-4o systems were withdrawn: they came from a hosted API that
# the methodology chapter records as a departure from the data use
# agreement, and no reported number in this thesis derives from them any
# more. The frontier rows replace them, read from the compliant run's own
# metrics artifact rather than from step3_baselines.json.
fm = json.loads(open(os.path.join(
    REPO, "results/accord/differential/"
          "frontier_anthropic_metrics.json")).read())
def _cells(strategy):
    c = fm["by_strategy"][strategy]["G_253"]
    return {"tp": c["tp"], "tn": c["tn"], "fp": c["fp"], "fn": c["fn"]}
systems = {
    "ACCORD": baselines["accord_binary_at_0.5_253core"],
    "Claude Sonnet 5 zero-shot": _cells("zero_shot"),
    "Claude Sonnet 5 few-shot": _cells("few_shot"),
    "Llama 3 8B": baselines["llama_baseline"],
}
ref = systems["ACCORD"]
TP0, TN0, FP0, FN0 = ref["tp"], ref["tn"], ref["fp"], ref["fn"]
P = TP0 + FN0
N = TN0 + FP0
mcc0 = mcc_from_counts(TP0, TN0, FP0, FN0)

fig, ax = plt.subplots(figsize=(FIGW, 3.39))
canvas(fig)
# The right margin is wide because the three slope labels live in it, and
# it is a fixed number of 9pt lines wide however big the canvas gets.
# right=0.69626, not the 0.7802 that holds the old margin absolute. The
# three slope labels sit in this margin and the longest of them measures
# 118.4pt; at 0.7802 it ran 35pt off the right edge of the canvas, savefig's
# tight box grew to catch it, and \includegraphics then scaled the whole
# figure DOWN by 8.3 percent to fit the measure -- which set this figure's
# "9pt" type at 8.3pt on the page, under the set's own floor. The margin now
# holds the longest label exactly, the canvas comes out at its nominal
# width, and the plot prints 9 percent larger than it did even though its
# drawn width is unchanged.
fig.subplots_adjust(left=0.1080, right=0.69626, top=0.8294, bottom=0.1372)
# Zoomed to the occupied wedge. The old range ran to x = 12 only to give the
# two long leaders somewhere to go, and the leaders are gone. Starting at a
# negative x is what pulls the boxed left spine off the origin cluster.
X_LO, X_HI = -1.2, 7.0
# Y_HI is the FRAME TOP, not the data maximum. set_yticks() below unions the
# view interval with its own ticks, so a 200 tick pulls the top of the box to
# 200 whatever set_ylim said; building the region polygons to a smaller
# number left a white strip across the plot under the frame top. One
# constant, used by the limits, the ticks and the fills alike.
Y_LO, Y_HI = -30.0, 200.0
ax.set_xlim(X_LO, X_HI)
ax.set_ylim(Y_LO, Y_HI)

d = np.linspace(0, X_HI, 200)
slope_ba = N / P
# The indifference lines are not measurements: they are what each statistic
# would accept, computed from the class counts. What a reader needs is the
# SPAN of what different statistics will trade, so the span is filled.
deltas = np.linspace(0, min(TP0, X_HI), 60)
dtn_curve = []
for delta in deltas:
    tp = TP0 - delta
    fn = P - tp

    def f(tn):
        fp = N - tn
        return mcc_from_counts(tp, tn, fp, fn) - mcc0

    try:
        dtn_curve.append(brentq(f, 0, N) - TN0)
    except ValueError:
        dtn_curve.append(np.nan)
dtn_curve = np.array(dtn_curve)

# Three regions, and between them they are the whole content of this
# figure. Above the balanced-accuracy slope every statistic on this canvas
# counts the trade as a gain; below the PABAK slope none of them does;
# between the two it depends entirely on which statistic was chosen, and
# that span is the thing a reader has to see. Wider region first, narrower
# over it. HAIR under MIST rather than MIST under something darker: there
# is no structural stop between MIST and SLATE, and SLATE at area scale is
# a mark colour, not a ground.
dfull = np.linspace(X_LO, X_HI, 200)
ax.fill_between(dfull, np.clip(dfull * slope_ba, Y_LO, Y_HI), Y_HI,
                facecolor=HAIR, edgecolor="none", linewidth=0, zorder=1)
ax.fill_between(dfull, Y_LO, np.clip(dfull, Y_LO, Y_HI), facecolor=HAIR,
                edgecolor="none", linewidth=0, zorder=1)
ax.fill_between(dfull, np.clip(dfull, Y_LO, Y_HI),
                np.clip(dfull * slope_ba, Y_LO, Y_HI), facecolor=MIST,
                edgecolor="none", linewidth=0, zorder=2)
case(ax.text(6.7, 26.0, "only some of these statistics\n"
                        "accept a trade in this wedge",
             ha="right", va="center", fontsize=T_NOTE, color=MUTE,
             linespacing=1.24, zorder=4))
ax.plot(d, d * slope_ba, color=GRAPHITE, lw=LW_REF, zorder=3)
ax.plot(d, d, color=GRAPHITE, lw=LW_REF, zorder=3)
ax.plot(deltas, dtn_curve, color=GRAPHITE, lw=LW_REF, zorder=3)
# The x = 0 reference, now clearly separate from the boxed left spine.
ax.axvline(0, color=SLATE, linestyle=REF_DASH, lw=LW_REF, zorder=2)

# Each slope is labelled at the right edge of the plot at its own exit y,
# horizontal. Where two exit y values fall within 11pt the lower one drops
# to the next slot and takes a 3pt tick at the frame; a tick at the frame
# is not a leader.
exits = [
    (min(X_HI * slope_ba, Y_HI), f"balanced accuracy (slope {slope_ba:.1f})"),
    (X_HI, "PABAK (slope 1)"),
    (float(dtn_curve[-1]), f"MCC through ACCORD ({mcc0:.3f})"),
]
exits.sort(key=lambda e: -e[0])
pt_per_y = (0.8294 - 0.1372) * 3.39 * 72 / (Y_HI - Y_LO)
placed_y = []
for y, text in exits:
    yy = y
    for prev in placed_y:
        if abs(yy - prev) * pt_per_y < 11.0:
            yy = prev - 11.0 / pt_per_y
    placed_y.append(yy)
    ax.plot([X_HI - 0.10, X_HI], [yy, yy], color=GRAPHITE, lw=1.0,
            clip_on=False, zorder=5)
    ax.text(X_HI + 0.18, yy, text, ha="left", va="center",
            fontsize=T_SERIES, color=GRAPHITE, clip_on=False, zorder=5)

# The four systems. ACCORD is the claim: this whole figure is about where
# ACCORD sits against exchange rates it never chose, so it is the one solid
# scarlet mark and it gives up its amber on this canvas.
spec = {
    "ACCORD": ("D", SCARLET_DEEP, SCARLET_CORE, SCARLET_DEEP, 7.0,
               (-6, 7), "right", "bottom"),
    "Llama 3 8B": ("D", PAPER, PLUM_CORE, PLUM_DEEP, MS_MARK,
                   (8, -6), "left", "top"),
    # EMERALD and CYAN, matching the hues fig-forest gives these two rows.
    # COBALT and AZURE belonged to the withdrawn GPT-4o systems and are not
    # reused, so a reader who learned the palette earlier is not misled.
    "Claude Sonnet 5 zero-shot": ("s", PAPER, EMERALD.CORE, EMERALD.DEEP,
                                  MS_MARK, (7, 4), "left", "bottom"),
    "Claude Sonnet 5 few-shot": ("^", PAPER, CYAN.CORE, CYAN.DEEP,
                                 MS_MARK, (7, -5), "left", "top"),
}
for name, s_ in systems.items():
    delta = TP0 - s_["tp"]
    dtn = s_["tn"] - TN0
    mk, face, edge, tone, ms, off, ha, va = spec[name]
    ax.plot([delta], [dtn], mk, markerfacecolor=face, markeredgecolor=edge,
            markeredgewidth=LW_CHIP if name == "ACCORD" else 1.2, ms=ms,
            zorder=6)
    case(ax.annotate(name, (delta, dtn), textcoords="offset points",
                     xytext=off, ha=ha, va=va, fontsize=T_SERIES,
                     color=tone, zorder=7))

ax.set_xlabel(r"$\delta$ = forgone true positives, relative to ACCORD")
ax.set_ylabel(r"$\Delta$TN, relative to ACCORD")
ax.set_xticks([0, 2, 4, 6])
ax.set_yticks([0, 50, 100, 150, Y_HI])
frame(ax)
signature(fig, SCARLET_DEEP,
          "Two of three systems clear every exchange rate ACCORD's "
          "counts define.")
fname = "fig-met-exchange.pdf"
fig.savefig(os.path.join(FIGDIR, fname))
plt.close(fig)
register(fname, "fig:met-exchange", [p_baselines],
         "Indifference lines are closed-form (PABAK, balanced accuracy) or numerically "
         "root-solved (MCC, scipy.optimize.brentq) from the confusion-matrix class counts "
         "in diag/step3_baselines.json; not fabricated curve shapes.")


## Final: verify all twelve exist


In [ ]:
EXPECTED = [
    "fig-pipeline-status.pdf",
    "fig-intro-headline.pdf",
    "fig-cohort-flow.pdf",
    "fig-exemplar.pdf",
    "fig-prevalence-response.pdf",
    "fig-falsification.pdf",
    "fig-power.pdf",
    "fig-forest.pdf",
    "fig-scaling.pdf",
    "fig-threshold-surface.pdf",
    "fig-disc-actionability.pdf",
    "fig-met-exchange.pdf",
]
missing = [f for f in EXPECTED if not os.path.exists(os.path.join(FIGDIR, f))]
print("figures written:", len(EXPECTED) - len(missing), "of", len(EXPECTED))
for f in missing:
    print("  MISSING", f)
assert not missing, missing
